# Unit 5. Variable Selection and Dimension Reduction

精算統計模型 (SOA Exam SRM/PA). The text of [unit05.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit05.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("ISLR2", "leaps", "pls")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/TermLife.csv", "data/frees/csv/UNLifeExpectancy.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2); library(leaps); library(pls)

## 1 Subset Selection

> **Reading.** ISLR §6.1.1–6.1.2 · FREES §5.2. SRM learning outcome 2(e).

**Remark** (The classical assumptions; Unit 2). A1–A5 are Unit 2's classical assumptions on $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ fixed: linearity (A1), zero-mean errors (A2), $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ with $\sigma^2>0$ (A3), $\operatorname{rank}(\mathbf{X})=k+1$, with $n>k+1$ for inference (A4), and normal errors (A5).

### 1.1 Search procedures

**Definition 5.1** (Candidate models; ISLR §6.1.1, FREES §5.2). Let $x_1,\ldots,x_p$ be the candidate predictors, a factor with $c$ levels contributing its $c-1$ indicator columns as separate candidates. For $S\subseteq\{1,\ldots,p\}$ with $|S|=d$, the model $\mathcal{M}_S$ is the least squares regression of $y$ on an intercept and the $x_j$ with $j\in S$: it has $d+1$ coefficients and the $n\times(d+1)$ design $\mathbf{X}_S$, and $\mathrm{RSS}(S)$ and $R^2(S)=1-\mathrm{RSS}(S)/\mathrm{TSS}$ are its residual sum of squares and coefficient of determination, with $\mathrm{TSS}=\sum_i(y_i-\bar y)^2>0$. The **null model** $\mathcal{M}_\emptyset$ fits $\bar y$. Whenever $\mathcal{M}_S$ is fitted, $\mathbf{X}_S$ is assumed to have full column rank.

**Definition 5.2** (Best subset, forward and backward stepwise selection; ISLR §6.1.1–6.1.2). Each procedure produces models $\mathcal{M}_0,\ldots,\mathcal{M}_p$, $\mathcal{M}_d$ having $d$ predictors, ties being broken arbitrarily.

(i) **Best subset selection**: $\mathcal{M}_d$ minimises $\mathrm{RSS}(S)$ over all $S$ with $|S|=d$.

(ii) **Forward stepwise selection**: $\mathcal{M}_0=\mathcal{M}_\emptyset$, and $\mathcal{M}_{d+1}$ adds to $\mathcal{M}_d$ the candidate that gives the enlarged model the smallest $\mathrm{RSS}$.

(iii) **Backward stepwise selection**: $\mathcal{M}_p$ contains all $p$ candidates, and $\mathcal{M}_{d-1}$ deletes from $\mathcal{M}_d$ the predictor whose removal leaves the smallest $\mathrm{RSS}$.

One of $\mathcal{M}_0,\ldots,\mathcal{M}_p$ is then chosen by an estimate of its test error. A **hybrid** procedure adds predictors as in (ii) and, after each addition, removes any predictor that no longer improves the fit.

**Definition 5.3** (Stepwise regression by $t$ ratios; FREES §5.2). Fix a cutoff $c>0$. **Forward selection** starts from the null model and repeatedly adds the candidate whose $t$ ratio in the enlarged model is largest in absolute value, provided it exceeds $c$, and stops when none does. **Backward selection** starts from the model with all candidates and repeatedly deletes the predictor whose $t$ ratio is smallest in absolute value, provided it is below $c$. The **stepwise regression algorithm** alternates the two: after each addition it deletes, one at a time, predictors whose $|t|$ has fallen below $c$, until no addition or deletion is possible. Replacing each $t$ ratio by its square, the partial $F$ ratio of Unit 3's corollary **extra sum of squares**, gives the same procedures.

**Proposition 5.4** (An $\mathrm{RSS}$ step is a $t$ step; FREES §5.2, ISLR §6.1.2). Let the current model have $d$ predictors, residual sum of squares $\mathrm{RSS}_d$ and $s^2=\mathrm{RSS}_d/(n-d-1)$.

(i) If $n>d+2$ and adding the candidate $x_j$ gives the residual sum of squares $\mathrm{RSS}_{+j}>0$, the $t$ ratio of $x_j$ in the enlarged model satisfies

$$
t_j^2=(n-d-2)\Bigl(\frac{\mathrm{RSS}_d}{\mathrm{RSS}_{+j}}-1\Bigr),
$$

so the candidate with the smallest $\mathrm{RSS}_{+j}$ is the candidate with the largest $|t_j|$.

(ii) If $n>d+1$, $d\geqslant1$ and $\mathrm{RSS}_d>0$, the $t$ ratio of a predictor $x_j$ of the current model satisfies $t_j^2=(\mathrm{RSS}_{-j}-\mathrm{RSS}_d)/s^2$, where $\mathrm{RSS}_{-j}$ is the residual sum of squares after deleting $x_j$; so the predictor whose deletion leaves the smallest $\mathrm{RSS}_{-j}$ is the one with the smallest $|t_j|$, and the largest $p$-value.

Forward and backward selection by $t$ ratios therefore add and delete predictors in the order of forward and backward stepwise selection, and differ from them only in stopping at the cutoff.

**Proof.** (i) By Unit 2's lemma **adding one explanatory variable**, applied with $\mathbf{z}=\mathbf{x}_j$ to the current design, $\mathrm{RSS}_d-\mathrm{RSS}_{+j}=\hat\gamma^2\,\mathbf{z}'\mathbf{M}\mathbf{z}$ and $\mathrm{SE}(\hat\gamma)^2=s_+^2/(\mathbf{z}'\mathbf{M}\mathbf{z})$ with $s_+^2=\mathrm{RSS}_{+j}/(n-d-2)$. Hence

$$
t_j^2=\frac{\hat\gamma^2}{\mathrm{SE}(\hat\gamma)^2}=\frac{\mathrm{RSS}_d-\mathrm{RSS}_{+j}}{s_+^2}
=(n-d-2)\Bigl(\frac{\mathrm{RSS}_d}{\mathrm{RSS}_{+j}}-1\Bigr).
$$

$\mathrm{RSS}_d$ is the same for every candidate, so $t_j^2$ is a strictly decreasing function of $\mathrm{RSS}_{+j}$.

(ii) Apply the same lemma to the model with $d-1$ predictors obtained by deleting $x_j$, enlarged by $\mathbf{z}=\mathbf{x}_j$ back to the current model, whose mean square error is $\mathrm{RSS}_d/\{n-(d-1)-2\}=s^2$: $\mathrm{RSS}_{-j}-\mathrm{RSS}_d=t_j^2s^2$, with $s^2$ common to all $j$. The $p$-value $2\operatorname{\mathsf{P}}(T>|t_j|)$, $T\sim t_{n-d-1}$, decreases as $|t_j|$ increases. ∎

**Proposition 5.5** (Counting and nesting; ISLR §6.1.1–6.1.2, FREES §5.2). (i) Best subset selection fits $2^p$ models; forward and backward stepwise selection fit $1+p(p+1)/2$ each.

(ii) The forward and the backward sequences are nested: $\mathcal{M}_d\subset\mathcal{M}_{d+1}$.

(iii) If $T\subseteq S$ then $\mathrm{RSS}(T)\geqslant\mathrm{RSS}(S)$. Hence a subset $S$ whose $\mathrm{RSS}(S)$ is at least the smallest $\mathrm{RSS}$ found so far among models of size $d\leqslant|S|$ contains no better model of size $d$.

(iv) For each $d$, the best subset $\mathcal{M}_d$ has $\mathrm{RSS}$ no larger than the forward and the backward $\mathcal{M}_d$, and in each of the three sequences $\mathrm{RSS}(\mathcal{M}_d)$ does not increase and $R^2(\mathcal{M}_d)$ does not decrease as $d$ grows.

(v) $\mathcal{M}_S$ has unique least squares coefficients only if $|S|+1\leqslant n$: forward stepwise selection can run to $\mathcal{M}_{n-1}$ when $p\geqslant n$, and backward stepwise selection needs $n\geqslant p+1$ to fit its first model.

**Proof.** (i) A subset is a choice, for each of the $p$ candidates, of whether it is in, so there are $2^p$ subsets. Forward selection fits the null model and then $p-d$ candidates at the step from $\mathcal{M}_d$, in all $1+\sum_{d=0}^{p-1}(p-d)=1+\sum_{j=1}^pj=1+p(p+1)/2$ models; backward selection fits the full model and then $d$ candidates at the step from $\mathcal{M}_d$, in all $1+\sum_{d=1}^pd$.

(ii) Each step of Definition 5.2(ii) adds one predictor, and each step of (iii) deletes one.

(iii) $\mathcal{C}(\mathbf{X}_T)\subseteq\mathcal{C}(\mathbf{X}_S)$, and Unit 2's proposition **bounds and monotonicity** shows that the residual sum of squares of the larger column space is no larger. A subset $T\subseteq S$ of size $d$ then has $\mathrm{RSS}(T)\geqslant\mathrm{RSS}(S)$, which is at least the best value found.

(iv) The forward and backward $\mathcal{M}_d$ are subsets of size $d$, over which the best subset $\mathcal{M}_d$ minimises. In the forward and backward sequences monotonicity follows from (ii) and (iii). For best subset selection and $d<p$, take $j\notin\mathcal{M}_d$: by (iii), $\mathrm{RSS}(\mathcal{M}_{d+1})\leqslant\mathrm{RSS}(\mathcal{M}_d\cup\{j\})\leqslant\mathrm{RSS}(\mathcal{M}_d)$. Finally $R^2=1-\mathrm{RSS}/\mathrm{TSS}$.

(v) $\mathbf{X}_S$ has $|S|+1$ columns and rank at most $n$, so it has full column rank only if $|S|+1\leqslant n$; without full rank the minimiser is not unique, by Unit 2's remark **without full rank**. ∎

**Remark** (ISLR §6.1.1). For models fitted by maximum likelihood, such as logistic regression, the **deviance**, $-2$ times the maximised log-likelihood, takes the place of $\mathrm{RSS}$ in Definition 5.2. Proposition 5.5(iii) is the **branch-and-bound** principle that lets best subset software skip subsets; ISLR puts the practical limit of best subset selection at about $p=40$.

**Example 5.1** (All subsets of the `Credit` predictors; ISLR §6.1.1–6.1.2, Figure 6.1, Table 6.1). The ten predictors of `Balance` give $p=11$ candidates, `Region` having three levels. Every subset is fitted, and `regsubsets` finds the same best subsets:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.5)
Xc <- model.matrix(Balance ~ ., Credit)[, -1]; yc <- Credit$Balance
n_c <- nrow(Xc); p_c <- ncol(Xc); tss_c <- sum((yc - mean(yc))^2)
subsets <- unlist(lapply(1:p_c, function(d) combn(p_c, d, simplify = FALSE)), recursive = FALSE)
rss_all <- sapply(subsets, function(S) sum(lm.fit(cbind(1, Xc[, S, drop = FALSE]), yc)$residuals^2))
d_all <- lengths(subsets); best_rss <- tapply(rss_all, d_all, min)
c(non_null_models = length(subsets), two_to_p = 2^p_c)
round(1 - best_rss / tss_c, 4)
b_c <- summary(regsubsets(Balance ~ ., data = Credit, nvmax = 11))
f_c <- summary(regsubsets(Balance ~ ., data = Credit, nvmax = 11, method = "forward"))
c(regsubsets_vs_all_fits = max(abs(b_c$rss - best_rss)))
vars <- function(w, d) paste(colnames(w)[-1][w[d, -1]], collapse = ", ")
for (d in 1:4) cat(d, "best:   ", vars(b_c$which, d), "\n  forward:", vars(f_c$which, d), "\n")
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.6, 0.6, 0.6), mgp = c(2, 0.6, 0))
plot(d_all, rss_all / 1e6, pch = 16, cex = 0.3, col = "grey50", xlab = "number of predictors",
     ylab = "RSS (millions)")
lines(1:p_c, best_rss / 1e6, col = "red", lwd = 1.5)
plot(d_all, 1 - rss_all / tss_c, pch = 16, cex = 0.3, col = "grey50", xlab = "number of predictors",
     ylab = expression(R^2))
lines(1:p_c, 1 - best_rss / tss_c, col = "red", lwd = 1.5)
par(op)

*Figure 5.1.* $\mathrm{RSS}$ and $R^2$ of the $2047$ non-null models for `Balance`; the red curves join the best model of each size (ISLR Figure 6.1).

The best one-, two- and three-predictor models are also the forward ones; at $d=4$ best subset selection drops `Rating` for `Limit` and `Cards`, which forward selection cannot do, and the best three-predictor model is not contained in the best four-predictor model, so best subset sequences need not be nested. The best $R^2$ is $0.7458$, $0.8751$ and $0.9499$ with one, two and three predictors and $0.9551$ with all eleven.

### 1.2 Data snooping

**Proposition 5.6** (False rejections; FREES §5.2). Let $m$ tests of true null hypotheses each reject with probability $\alpha$, and let $N$ be the number of rejections. Then $\operatorname{\mathsf{E}}[N]=m\alpha$ and $\operatorname{\mathsf{P}}(N\geqslant1)\leqslant m\alpha$, whatever the dependence among the tests; if the tests are independent, $\operatorname{\mathsf{P}}(N\geqslant1)=1-(1-\alpha)^m$.

**Proof.** $N=\sum_{j=1}^m\mathbb{1}_{A_j}$ with $A_j$ the event that test $j$ rejects, so $\operatorname{\mathsf{E}}[N]=\sum_j\operatorname{\mathsf{P}}(A_j)=m\alpha$ by linearity. $\{N\geqslant1\}=\bigcup_jA_j$ has probability at most $\sum_j\operatorname{\mathsf{P}}(A_j)$ by subadditivity. Under independence $\operatorname{\mathsf{P}}(N=0)=\prod_j\operatorname{\mathsf{P}}(A_j^c)=(1-\alpha)^m$. ∎

**Example 5.2** (Data snooping in stepwise regression; FREES §5.2). FREES's chapter 5 script draws $n=100$ responses and $50$ candidates, all independent standard normal, so every slope is zero. It fits all $50$, then runs the stepwise algorithm with the BIC criterion (R's `step` with $k=\ln n$); the $t$-ratio algorithm of Definition 5.3 is run with FREES's two cutoffs:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
set.seed(1237)
snoop <- as.data.frame(matrix(rnorm(100 * 50), ncol = 50))
colnames(snoop) <- paste0("obs", 1:50)
snoop$y <- rnorm(100)
fit_summary <- function(m) { sm <- summary(m); f <- sm$fstatistic
  c(s = sm$sigma, R2 = sm$r.squared, F = f[[1]], df1 = f[[2]], df2 = f[[3]],
    p_value = pf(f[[1]], f[[2]], f[[3]], lower.tail = FALSE)) }
full_s <- lm(y ~ ., data = snoop)
round(fit_summary(full_s), 4)
step_s <- step(full_s, k = log(100), trace = 0)
formula(step_s)
round(fit_summary(step_s), 4)
t_step <- function(dat, cut) {
  cand_all <- setdiff(names(dat), "y"); inm <- character(0)
  repeat {
    cand <- setdiff(cand_all, inm)
    tt <- sapply(cand, function(v) abs(coef(summary(lm(reformulate(c(inm, v), "y"), data = dat)))[v, 3]))
    if (max(tt) <= cut) break
    inm <- c(inm, cand[which.max(tt)])
    while (length(inm) > 1) {
      ti <- abs(coef(summary(lm(reformulate(inm, "y"), data = dat)))[inm, 3])
      if (min(ti) >= cut) break
      inm <- inm[-which.min(ti)]
    }
  }
  inm
}
for (cut in c(2, 1.645)) {
  v <- t_step(snoop, cut); cat("cutoff", cut, ":", v, "\n")
  print(round(fit_summary(lm(reformulate(v, "y"), data = snoop)), 4))
}

The fit on all $50$ candidates has $F=1.2642$ on $(50,49)$ degrees of freedom and $p=0.2066$, in agreement with the null. BIC stepwise selection and the $t$-ratio algorithm with cutoff $2$ both keep `obs27`, `obs29` and `obs32`, whose overall $F=5.2124$ has $p=0.0022$; with cutoff $1.645$ seven candidates are kept, with $F=4.5931$ and $p=0.0002$. FREES's printed values ($s=1.142$, $R^2=46.2\%$, $F=0.84$; two variables kept with $F=5.09$ and six with $F=4.61$) come from a different draw. At the first step alone $50$ tests are run at level $0.05$, and $2.5$ false rejections are expected by Proposition 5.6. The $F$ test of a selected model rejects far more often than its level; over $1000$ such data sets, with forward selection and BIC:

In [ ]:
set.seed(2026)
mc_snoop <- t(replicate(1000, {
  Xm <- matrix(rnorm(100 * 50), ncol = 50); ym <- rnorm(100)
  fw <- summary(regsubsets(x = Xm, y = ym, nvmax = 10, method = "forward"))
  crit <- c(100 * log(sum((ym - mean(ym))^2)), 100 * log(fw$rss) + log(100) * (1:10))
  d <- which.min(crit) - 1
  p_F <- if (d == 0) 1 else {
    f <- summary(lm(ym ~ Xm[, fw$which[d, -1], drop = FALSE]))$fstatistic
    pf(f[[1]], f[[2]], f[[3]], lower.tail = FALSE) }
  c(size = d, reject = p_F < 0.05)
}))
colMeans(mc_snoop)

The selected models have $2.365$ predictors on average, and their $F$ tests reject at the nominal $5\%$ level in a proportion $0.854$ of the data sets, although every null hypothesis is true.

**Remark** (Drawbacks of stepwise regression; FREES §5.2). FREES lists seven: (1) the search snoops through many models and fits the data too well, so the $t$ and $F$ ratios of the chosen model overstate the importance of its variables (Example 5.2); (2) the chosen model need not be the best, and nonlinear terms, outliers and high leverage points are ignored; (3) not all $2^p$ models are searched; (4) a single criterion is used; (5) the significance level of a sequence of tests has no meaning; (6) predictors are judged one at a time, so jointly important ones, such as the suppressor variables of Unit 4, can be missed; FREES notes that backward selection, starting from all of them, can retain them; (7) the investigator's knowledge is not used, which is remedied by forcing variables into the model. Best subset selection answers (3). For (5) FREES advises a cutoff below $2$, such as $1.645$ or even $1$; by Unit 2's theorem **adjusted $R^2$ rises iff $|t|>1$**, a variable with $|t|>1$ lowers $s$. A model chosen automatically is subject to the same diagnostic checking as any other.

## 2 Selection Criteria

> **Reading.** ISLR §6.1.3 · FREES §5.6, §6.2. SRM learning outcome 2(e).

### 2.1 The optimism of the training error

**Definition 5.7** (Training error and in-sample prediction error; ISLR §6.1.3). Let $\mathbf{Y}=\boldsymbol{\mu}+\boldsymbol{\varepsilon}$ with $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}]=\mathbf{0}$, $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ and $\boldsymbol{\mu}\in\mathbb{R}^n$ arbitrary, and let a model with an $n\times q$ design $\mathbf{X}_1$ of rank $q<n$ be fitted by least squares, with hat matrix $\mathbf{H}_1$, residual maker $\mathbf{M}_1=\mathbf{I}_n-\mathbf{H}_1$, fitted values $\hat{\mathbf{Y}}_1=\mathbf{H}_1\mathbf{Y}$ and $\mathrm{RSS}_1=\|\mathbf{Y}-\hat{\mathbf{Y}}_1\|^2=\mathbf{Y}'\mathbf{M}_1\mathbf{Y}$. Let $\mathbf{Y}^0=\boldsymbol{\mu}+\boldsymbol{\varepsilon}^0$ be new responses at the same design points, $\boldsymbol{\varepsilon}^0$ being independent of $\boldsymbol{\varepsilon}$ and distributed as $\boldsymbol{\varepsilon}$. The **training error** is $\mathrm{RSS}_1/n$, and the **in-sample prediction error** is $\mathrm{Err}_{\mathrm{in}}=n^{-1}\operatorname{\mathsf{E}}\|\mathbf{Y}^0-\hat{\mathbf{Y}}_1\|^2$.

**Theorem 5.8** (The optimism of the training error; ISLR §6.1.3). Under Definition 5.7,

(i) $\operatorname{\mathsf{E}}[\mathrm{RSS}_1]=(n-q)\sigma^2+\|\mathbf{M}_1\boldsymbol{\mu}\|^2$;

(ii) $n\mathrm{Err}_{\mathrm{in}}=(n+q)\sigma^2+\|\mathbf{M}_1\boldsymbol{\mu}\|^2$;

(iii) $n\mathrm{Err}_{\mathrm{in}}-\operatorname{\mathsf{E}}[\mathrm{RSS}_1]=2q\sigma^2$, whether or not $\boldsymbol{\mu}\in\mathcal{C}(\mathbf{X}_1)$.

**Proof.** **Step 1: two traces.** $\mathbf{H}_1$ is symmetric and idempotent with column space $\mathcal{C}(\mathbf{X}_1)$, so $\operatorname{tr}(\mathbf{H}_1)=\operatorname{rank}(\mathbf{H}_1)=q$ by the prerequisites' lemma **trace equals rank**, and $\operatorname{tr}(\mathbf{M}_1)=n-q$.

**Step 2: the training error.** By the prerequisites' lemma **expectation of a quadratic form**, with $\operatorname{\mathsf{E}}[\mathbf{Y}]=\boldsymbol{\mu}$ and $\operatorname{var}(\mathbf{Y})=\sigma^2\mathbf{I}_n$, $\operatorname{\mathsf{E}}[\mathbf{Y}'\mathbf{M}_1\mathbf{Y}]=\sigma^2\operatorname{tr}(\mathbf{M}_1)+\boldsymbol{\mu}'\mathbf{M}_1\boldsymbol{\mu}=(n-q)\sigma^2+\|\mathbf{M}_1\boldsymbol{\mu}\|^2$, since $\boldsymbol{\mu}'\mathbf{M}_1\boldsymbol{\mu}=\boldsymbol{\mu}'\mathbf{M}_1'\mathbf{M}_1\boldsymbol{\mu}$.

**Step 3: the prediction error.** Write $\mathbf{Y}^0-\hat{\mathbf{Y}}_1=\boldsymbol{\varepsilon}^0+(\boldsymbol{\mu}-\mathbf{H}_1\mathbf{Y})$. The second term is a function of $\boldsymbol{\varepsilon}$, hence independent of $\boldsymbol{\varepsilon}^0$, and $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}^0]=\mathbf{0}$, so the expected cross product vanishes and

$$
\operatorname{\mathsf{E}}\|\mathbf{Y}^0-\hat{\mathbf{Y}}_1\|^2=\operatorname{\mathsf{E}}\|\boldsymbol{\varepsilon}^0\|^2+\operatorname{\mathsf{E}}\|\boldsymbol{\mu}-\mathbf{H}_1\mathbf{Y}\|^2
=n\sigma^2+\operatorname{\mathsf{E}}\|\boldsymbol{\mu}-\mathbf{H}_1\mathbf{Y}\|^2 .
$$

Now $\boldsymbol{\mu}-\mathbf{H}_1\mathbf{Y}=\mathbf{M}_1\boldsymbol{\mu}-\mathbf{H}_1\boldsymbol{\varepsilon}$, and the cross term $-2\boldsymbol{\mu}'\mathbf{M}_1\mathbf{H}_1\boldsymbol{\varepsilon}$ is identically zero because $\mathbf{M}_1\mathbf{H}_1=\mathbf{0}$. So $\|\boldsymbol{\mu}-\mathbf{H}_1\mathbf{Y}\|^2=\|\mathbf{M}_1\boldsymbol{\mu}\|^2+\boldsymbol{\varepsilon}'\mathbf{H}_1\boldsymbol{\varepsilon}$, whose expectation is $\|\mathbf{M}_1\boldsymbol{\mu}\|^2+\sigma^2\operatorname{tr}(\mathbf{H}_1)=\|\mathbf{M}_1\boldsymbol{\mu}\|^2+q\sigma^2$ by the same lemma. This gives (ii), and subtracting (i) gives (iii). ∎

**Corollary 5.9** (What a column costs and what it buys; FREES §6.2). Let $\mathcal{C}(\mathbf{X}_1)\subseteq\mathcal{C}(\mathbf{X}_2)$ with ranks $q_1<q_2<n$, and let $\mathrm{Err}_1$ and $\mathrm{Err}_2$ be the in-sample prediction errors of the two fits. Then $\|\mathbf{M}_2\boldsymbol{\mu}\|^2\leqslant\|\mathbf{M}_1\boldsymbol{\mu}\|^2$ and

$$
n(\mathrm{Err}_2-\mathrm{Err}_1)=(q_2-q_1)\sigma^2-\bigl(\|\mathbf{M}_1\boldsymbol{\mu}\|^2-\|\mathbf{M}_2\boldsymbol{\mu}\|^2\bigr).
$$

If $\boldsymbol{\mu}\in\mathcal{C}(\mathbf{X}_1)$, every extra column raises $\mathrm{Err}_{\mathrm{in}}$ by $\sigma^2/n$; in general the larger model predicts better exactly when its reduction of the squared bias exceeds $(q_2-q_1)\sigma^2$.

**Proof.** Subtract the two instances of Theorem 5.8(ii). $\mathbf{H}_2\boldsymbol{\mu}$ is the point of $\mathcal{C}(\mathbf{X}_2)$ closest to $\boldsymbol{\mu}$ by the prerequisites' theorem **orthogonal projection**, and $\mathbf{H}_1\boldsymbol{\mu}\in\mathcal{C}(\mathbf{X}_1)\subseteq\mathcal{C}(\mathbf{X}_2)$, so $\|\mathbf{M}_2\boldsymbol{\mu}\|\leqslant\|\mathbf{M}_1\boldsymbol{\mu}\|$. If $\boldsymbol{\mu}\in\mathcal{C}(\mathbf{X}_1)$, both norms are zero. ∎

**Remark** (FREES §6.2). Unit 3's theorem **omitted variables** and proposition **extraneous variables** describe the coefficients; Corollary 5.9 describes prediction. An extraneous column costs exactly $\sigma^2/n$ of in-sample prediction error, while an omitted one costs its contribution to $\|\mathbf{M}_1\boldsymbol{\mu}\|^2/n$, which is not bounded; hence FREES's advice, when in doubt, to leave the variable in.

### 2.2 Criteria that adjust the training error

**Definition 5.10** ($C_p$, AIC, BIC and adjusted $R^2$; ISLR §6.1.3). For a least squares model with an intercept, $d$ predictors and residual sum of squares $\mathrm{RSS}$, and an estimate $\hat\sigma^2>0$ of $\sigma^2$, usually the mean square error of the model with all $p$ candidates,

$$
C_p=\mathrm{AIC}=\frac{\mathrm{RSS}+2d\hat\sigma^2}{n},\qquad
\mathrm{BIC}=\frac{\mathrm{RSS}+\ln(n)\,d\,\hat\sigma^2}{n},\qquad
R_a^2=1-\frac{\mathrm{RSS}/(n-d-1)}{\mathrm{TSS}/(n-1)} .
$$

Small $C_p$, AIC and BIC and large $R_a^2$ are preferred. These are the forms the SRM syllabus prescribes.

**Remark** (ISLR §6.1.3). ISLR ranks the four: $C_p$, AIC and BIC have rigorous theoretical justifications, while adjusted $R^2$, intuitive as it is, is not as well motivated in statistical theory.

**Definition 5.11** (FREES's criteria; FREES §5.6.1). With $s^2=\mathrm{RSS}/(n-d-1)$ and $s_{\mathrm{full}}^2$ the mean square error of the model with all candidates,

$$
\mathrm{AIC}_{\mathrm F}=n\ln s^2+n\ln(2\pi)+n+3+d,\qquad
C_p^{\mathrm F}=\frac{\mathrm{RSS}}{s_{\mathrm{full}}^2}-n+2(d+1),
$$

FREES writing $p$ for the number $d+1$ of coefficients; $s$, $R^2$ and $R_a^2$ complete FREES's list of goodness-of-fit statistics.

**Theorem 5.12** ($C_p$ estimates the in-sample prediction error; ISLR §6.1.3). Under Definition 5.7, let the model have an intercept and $d$ predictors, so that $q=d+1$, and let $\operatorname{\mathsf{E}}[\hat\sigma^2]=\sigma^2$. Then

$$
\operatorname{\mathsf{E}}\Bigl[\frac{\mathrm{RSS}+2(d+1)\hat\sigma^2}{n}\Bigr]=\mathrm{Err}_{\mathrm{in}},\qquad
\operatorname{\mathsf{E}}[C_p]=\mathrm{Err}_{\mathrm{in}}-\frac{2\sigma^2}{n}.
$$

If $n>p+1$, the mean square error $\hat\sigma^2$ of the model with all $p$ candidates satisfies $\operatorname{\mathsf{E}}[\hat\sigma^2]=\sigma^2+\|\mathbf{M}_{\mathrm{full}}\boldsymbol{\mu}\|^2/(n-p-1)$, which equals $\sigma^2$ when $\boldsymbol{\mu}$ lies in that model's column space and exceeds it otherwise.

**Proof.** By linearity and Theorem 5.8(iii) with $q=d+1$, $\operatorname{\mathsf{E}}[\mathrm{RSS}+2(d+1)\hat\sigma^2]=\operatorname{\mathsf{E}}[\mathrm{RSS}]+2(d+1)\sigma^2=n\mathrm{Err}_{\mathrm{in}}$; $C_p$ has $2d$ in place of $2(d+1)$. For the full model, $\hat\sigma^2=\mathrm{RSS}_{\mathrm{full}}/(n-p-1)$ and Theorem 5.8(i) with $q=p+1$ gives its expectation; the added term is zero iff $\mathbf{M}_{\mathrm{full}}\boldsymbol{\mu}=\mathbf{0}$, that is iff $\boldsymbol{\mu}$ lies in the column space. ∎

**Remark** (ISLR §6.1.3). The offset $-2\sigma^2/n$ is the same for every model, so it does not change which model has the smallest $C_p$; ISLR's statement that $C_p$ is unbiased for the test error holds exactly when $d$ counts the intercept. The prediction error estimated is at the design points of the sample. The criteria of Definition 5.10 adjust the training error, and so estimate the test error indirectly; a validation set or cross-validation estimates it directly.

**Proposition 5.13** (One criterion in several forms; ISLR §6.1.3, FREES §5.6.1). Let the models have an intercept, $d$ predictors and $\mathrm{RSS}>0$, with $n>d+1$, let $\hat\sigma^2>0$ be the same for all of them, and let $\ell(\boldsymbol{\beta},\sigma^2)$ be the normal log-likelihood of Unit 2's proposition **least squares is maximum likelihood**, $\hat{\boldsymbol{\beta}}$ the least squares estimate.

(i) **Mallows' form** $C_p'=\mathrm{RSS}/\hat\sigma^2+2d-n$ satisfies $C_p=\hat\sigma^2(C_p'+n)/n$, and $C_p^{\mathrm F}$ with $s_{\mathrm{full}}^2=\hat\sigma^2$ equals $C_p'+2$.

(ii) With $\sigma^2$ fixed at $\hat\sigma^2$, $-2\ell(\hat{\boldsymbol{\beta}},\hat\sigma^2)+2d=(n/\hat\sigma^2)\,C_p+n\ln(2\pi\hat\sigma^2)$ and $-2\ell(\hat{\boldsymbol{\beta}},\hat\sigma^2)+\ln(n)\,d=(n/\hat\sigma^2)\mathrm{BIC}+n\ln(2\pi\hat\sigma^2)$.

(iii) With $\sigma^2$ at its maximum likelihood value $\tilde\sigma^2=\mathrm{RSS}/n$, $-2\ell(\hat{\boldsymbol{\beta}},\tilde\sigma^2)=n\ln(2\pi\mathrm{RSS}/n)+n$.

(iv) $\mathrm{AIC}_{\mathrm F}=-2\ell(\hat{\boldsymbol{\beta}},s^2)+2(d+2)$, and

$$
\mathrm{AIC}_{\mathrm F}-\bigl\{-2\ell(\hat{\boldsymbol{\beta}},\tilde\sigma^2)+2(d+2)\bigr\}
      =n\ln\frac{n}{n-d-1}-(d+1),
$$

which is positive and increasing in $d$.

**Proof.** (i) $\hat\sigma^2(C_p'+n)/n=(\mathrm{RSS}+2d\hat\sigma^2)/n$, and $C_p^{\mathrm F}-C_p'=\{-n+2(d+1)\}-\{2d-n\}=2$.

(ii) By Unit 2's log-likelihood, $-2\ell(\hat{\boldsymbol{\beta}},\sigma^2)=n\ln(2\pi\sigma^2)+\mathrm{RSS}/\sigma^2$. At $\sigma^2=\hat\sigma^2$, adding $2d$ gives $n\ln(2\pi\hat\sigma^2)+(\mathrm{RSS}+2d\hat\sigma^2)/\hat\sigma^2$, and adding $\ln(n)\,d$ gives the second identity.

(iii) Substitute $\sigma^2=\mathrm{RSS}/n$: $\mathrm{RSS}/\sigma^2=n$.

(iv) At $\sigma^2=s^2$, $\mathrm{RSS}/s^2=n-d-1$, so $-2\ell(\hat{\boldsymbol{\beta}},s^2)+2(d+2)=n\ln s^2+n\ln(2\pi)+n-d-1+2d+4=\mathrm{AIC}_{\mathrm F}$. Subtracting the expression of (iii) plus $2(d+2)$ leaves $n\ln\{s^2/(\mathrm{RSS}/n)\}+(n+3+d)-(n+2d+4)=n\ln\{n/(n-d-1)\}-(d+1)$. This is $n\,g(x)$ with $x=(d+1)/n\in(0,1)$ and $g(x)=-\ln(1-x)-x$; $g(0)=0$ and $g'(x)=x/(1-x)>0$ on $(0,1)$, so by the prerequisites' mean value theorem $g$ is positive and increasing there, and $x$ increases with $d$. ∎

**Remark** (Software forms). R's `AIC` and `BIC` applied to an `lm` fit add $2(d+2)$ and $\ln(n)(d+2)$ to the expression of Proposition 5.13(iii), counting $\sigma^2$ as a parameter; `step` ranks models by $n\ln(\mathrm{RSS}/n)+k(d+1)$, with $k=2$ by default and $k=\ln n$ for the BIC form; `regsubsets` reports $C_p^{\mathrm F}$ and $n\ln(\mathrm{RSS}/\mathrm{TSS})+(d+1)\ln n$.

**Proposition 5.14** (How the criteria compare; ISLR §6.1.3, FREES §5.2, §5.6.1). Let $\mathrm{RSS}_d$ be the residual sum of squares of the candidate with $d$ predictors in a sequence $\mathcal{M}_0,\ldots,\mathcal{M}_p$, and $\hat\sigma^2>0$.

(i) Among models with the same number $d$ of predictors, $\mathrm{RSS}$, $s^2$, $C_p$, AIC, BIC, $C_p'$, $C_p^{\mathrm F}$, $\mathrm{AIC}_{\mathrm F}$ and the forms of the remark on software forms increase with $\mathrm{RSS}$, and $R^2$ and $R_a^2$ decrease with it: all rank those models identically.

(ii) For $c>0$ let $d_c$ minimise $\mathrm{RSS}_d+c\,d\,\hat\sigma^2$, or $n\ln\mathrm{RSS}_d+c\,d$ when every $\mathrm{RSS}_d>0$, over $d$. If $c_1<c_2$, then $d_{c_2}\leqslant d_{c_1}$ for every choice of the minimisers. In particular, for $n\geqslant8$ BIC selects no more predictors than $C_p$ and AIC.

(iii) $R_a^2$ is largest exactly where $s^2$ is smallest; and if $n>d+2$, $\mathrm{TSS}>0$ and the enlarged model has a positive residual sum of squares, adding one predictor to $\mathcal{M}_d$ raises $R_a^2$ iff its $t$ ratio in the enlarged model has $|t|>1$.

(iv) Adding one predictor to $\mathcal{M}_d$, with $t$ its $t$ ratio and $s_+^2>0$ the mean square error of the enlarged model, lowers $\mathrm{RSS}+c\,d\,\hat\sigma^2$ iff $t^2>c\,\hat\sigma^2/s_+^2$, and lowers $n\ln\mathrm{RSS}+c\,d$ iff $t^2>(n-d-2)(e^{c/n}-1)$.

**Proof.** (i) For fixed $d$, $n$, $\hat\sigma^2$ and $\mathrm{TSS}$: $s^2$, $C_p$, AIC, BIC, $C_p'$ and $C_p^{\mathrm F}$ are increasing affine functions of $\mathrm{RSS}$; $\mathrm{AIC}_{\mathrm F}$, $n\ln(\mathrm{RSS}/n)+k(d+1)$, $n\ln(\mathrm{RSS}/\mathrm{TSS})+(d+1)\ln n$ and the forms of Proposition 5.13(iii) are increasing functions of $\mathrm{RSS}$; $R^2=1-\mathrm{RSS}/\mathrm{TSS}$ and $R_a^2=1-(n-1)\mathrm{RSS}/\{(n-d-1)\mathrm{TSS}\}$ are decreasing ones.

(ii) Write $d_1=d_{c_1}$, $d_2=d_{c_2}$ and $f_c(d)=\mathrm{RSS}_d+c\,d\,\hat\sigma^2$. By optimality $f_{c_1}(d_1)\leqslant f_{c_1}(d_2)$ and $f_{c_2}(d_2)\leqslant f_{c_2}(d_1)$. Adding the two inequalities and cancelling $\mathrm{RSS}_{d_1}+\mathrm{RSS}_{d_2}$, $(c_1d_1+c_2d_2)\hat\sigma^2\leqslant(c_1d_2+c_2d_1)\hat\sigma^2$, that is $(c_2-c_1)(d_2-d_1)\hat\sigma^2\leqslant0$, so $d_2\leqslant d_1$. The same argument applies to $n\ln\mathrm{RSS}_d+c\,d$. BIC has $c=\ln n$ and $C_p$ and AIC have $c=2$, and $\ln n>2$ for $n\geqslant8$ because $e^2<2.72^2<8$.

(iii) $R_a^2=1-s^2(n-1)/\mathrm{TSS}$. The second statement is Unit 2's theorem **adjusted $R^2$ rises iff $|t|>1$**.

(iv) By Proposition 5.4(i), $\mathrm{RSS}_d-\mathrm{RSS}_+=t^2s_+^2$ and $\mathrm{RSS}_d/\mathrm{RSS}_+=1+t^2/(n-d-2)$. The first criterion falls iff $\mathrm{RSS}_++c(d+1)\hat\sigma^2<\mathrm{RSS}_d+c\,d\,\hat\sigma^2$, iff $\mathrm{RSS}_d-\mathrm{RSS}_+>c\,\hat\sigma^2$, iff $t^2>c\,\hat\sigma^2/s_+^2$. The second falls iff $n\ln(\mathrm{RSS}_d/\mathrm{RSS}_+)>c$, iff $1+t^2/(n-d-2)>e^{c/n}$. ∎

**Remark** (FREES §5.6.1). When $\hat\sigma^2$ is close to $s_+^2$, $C_p$ and AIC admit a predictor whose $|t|$ exceeds about $\sqrt2$, BIC one whose $|t|$ exceeds about $\sqrt{\ln n}$, and $R_a^2$ one with $|t|>1$: $R_a^2$ penalises size least. FREES describes BIC as giving the penalty a smaller weight than AIC; by Proposition 5.14(ii) its weight $\ln n$ is the larger once $n\geqslant8$, and it chooses the smaller models.

**Proposition 5.15** (Likelihood ratio and $F$; SRM learning outcome 2(e)). Let a restricted model with $\mathrm{RSS}_R$ be nested in an unrestricted one with $k$ predictors, $\mathrm{RSS}_U>0$ and $q$ more coefficients, and let $F$ be Unit 3's statistic $\{(\mathrm{RSS}_R-\mathrm{RSS}_U)/q\}/\{\mathrm{RSS}_U/(n-k-1)\}$. The likelihood ratio statistic, twice the difference of the maximised normal log-likelihoods, is

$$
\mathrm{LR}=n\ln\frac{\mathrm{RSS}_R}{\mathrm{RSS}_U}=n\ln\Bigl(1+\frac{q\,F}{n-k-1}\Bigr),
$$

a strictly increasing function of $F$; so the test that rejects for large $\mathrm{LR}$ at the exact level of Unit 3 is the $F$ test. R's `AIC` prefers the unrestricted model iff $\mathrm{LR}>2q$, and `BIC` iff $\mathrm{LR}>q\ln n$.

**Proof.** By Unit 2's proposition **least squares is maximum likelihood** each maximum is at $(\hat{\boldsymbol{\beta}},\tilde\sigma^2)$, so by Proposition 5.13(iii) $2(\ell_U-\ell_R)=n\ln(2\pi\mathrm{RSS}_R/n)-n\ln(2\pi\mathrm{RSS}_U/n)=n\ln(\mathrm{RSS}_R/\mathrm{RSS}_U)$, and $\mathrm{RSS}_R/\mathrm{RSS}_U=1+qF/(n-k-1)$ by the definition of $F$. The unrestricted model has the smaller `AIC` iff $-2\ell_U+2(k+2)<-2\ell_R+2(k+2-q)$, iff $\mathrm{LR}>2q$; with $\ln n$ in place of $2$ for `BIC`. ∎

### 2.3 Validation and cross-validation

**Definition 5.16** (Out-of-sample validation and cross-validation for selection; FREES §5.6.2, ISLR §6.1.3). **Out-of-sample validation** splits the $n$ observations at random into a **model development** (training) subsample of size $n_1$ and a **validation** subsample of size $n_2$, fits each candidate to the first, and chooses the candidate with the smallest sum of squared prediction errors $\text{SSPE}=\sum_{i\text{ validated}}(y_i-\hat y_i)^2$. When the candidates are the $\mathcal{M}_d$ of Definition 5.2, the search itself is run on the training subsample. In $K$-fold cross-validation the search is run on each training fold, the held-out errors $\mathrm{MSE}_{jd}$ of the size-$d$ model of fold $j$ are averaged into $\mathrm{CV}_d=K^{-1}\sum_j\mathrm{MSE}_{jd}$, the size $\hat d$ minimising $\mathrm{CV}_d$ is chosen, and the best model of size $\hat d$ is found on all the data.

**Remark** (FREES §5.6.2). $\text{SSPE}$ depends on the random split and on the choice of $n_1$ and $n_2$. FREES reports Snee's advice not to split unless $n\geqslant2(k+1)+20$, and the rules of thumb of validating on $25$–$35\%$ of samples of $100$ or fewer and on half of samples of $500$ or more.

**Proposition 5.17** (PRESS; FREES §5.6.3). The predicted residual sum of squares $\text{PRESS}=\sum_i(y_i-\hat y_{(i)})^2$, with $\hat y_{(i)}$ the prediction of $y_i$ from the fit without observation $i$, equals $\sum_i\{e_i/(1-h_{ii})\}^2=n\,\mathrm{CV}_{(n)}$, where $e_i$ and $h_{ii}<1$ are the residual and leverage of observation $i$ in the fit on all $n$ observations.

**Proof.** This is Unit 1's proposition **LOOCV shortcut for least squares** multiplied by $n$. ∎

**Definition 5.18** (One-standard-error rule; ISLR §6.1.3). With $K$-fold cross-validation, let $\mathrm{SE}_d$ be the standard deviation of $\mathrm{MSE}_{1d},\ldots,\mathrm{MSE}_{Kd}$ divided by $\sqrt K$. The **one-standard-error rule** chooses the smallest $d$ with $\mathrm{CV}_d\leqslant\mathrm{CV}_{\hat d}+\mathrm{SE}_{\hat d}$.

**Remark**. $\mathrm{SE}_d$ treats the $K$ fold errors as uncorrelated. They share training observations, and by Unit 1's proposition **mean of correlated estimates**, if they have a common variance $\tau^2$ and a common positive correlation $\rho$, the variance of their mean is $\rho\tau^2+(1-\rho)\tau^2/K$, larger than $\tau^2/K$; $\mathrm{SE}_d$ sets a scale for the rule, not an exact standard error.

### 2.4 The criteria on the Credit data

**Example 5.3** ($C_p$, BIC and adjusted $R^2$; ISLR §6.1.3, Figure 6.2). For the best models of Example 5.1, with $\hat\sigma^2$ the mean square error of the model with all eleven candidates:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
s2_c <- summary(lm(Balance ~ ., data = Credit))$sigma^2; d_c <- 1:11
cp_c <- (b_c$rss + 2 * d_c * s2_c) / n_c; bic_c <- (b_c$rss + log(n_c) * d_c * s2_c) / n_c
c(sigma2_hat = s2_c, Cp_min = which.min(cp_c), BIC_min = which.min(bic_c),
  adjR2_max = which.max(b_c$adjr2))
for (d in c(4, 6, 7)) cat(d, ":", vars(b_c$which, d), "\n")
signif(c(regsubsets_cp = max(abs(b_c$cp - (b_c$rss / s2_c - n_c + 2 * (d_c + 1)))),
  regsubsets_bic = max(abs(b_c$bic - (n_c * log(b_c$rss / tss_c) + (d_c + 1) * log(n_c)))),
  regsubsets_adjr2 = max(abs(b_c$adjr2 - (1 - (b_c$rss / (n_c - d_c - 1)) / (tss_c / (n_c - 1)))))), 3)
c(regsubsets_bic_min = which.min(b_c$bic), sqrt_2 = sqrt(2), sqrt_log_n = sqrt(log(n_c)))
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.8, 0.6, 0.4), mgp = c(2.2, 0.6, 0))
for (v in list(list(cp_c, "Cp", which.min), list(bic_c, "BIC", which.min),
               list(b_c$adjr2, "adjusted R-squared", which.max))) {
  plot(d_c, v[[1]], type = "b", pch = 16, cex = 0.6, xlab = "number of predictors", ylab = v[[2]])
  points(v[[3]](v[[1]]), v[[1]][v[[3]](v[[1]])], col = "blue", pch = 4, cex = 2, lwd = 2)
}
par(op)

*Figure 5.2.* $C_p$, BIC and $R_a^2$ of the best `Credit` model of each size; crosses mark the selected sizes (ISLR Figure 6.2).

$C_p$ selects the six predictors `Income`, `Limit`, `Rating`, `Cards`, `Age` and `StudentYes`; BIC the four `Income`, `Limit`, `Cards` and `StudentYes`; $R_a^2$ seven, adding `OwnYes`. These are ISLR's selections: BIC chooses no more predictors than $C_p$, as Proposition 5.14(ii) requires, and $R_a^2$, whose threshold is the weakest, the most. `regsubsets` reports $C_p^{\mathrm F}$, which ranks the models as $C_p$ does by Proposition 5.13(i), and the form $n\ln(\mathrm{RSS}/\mathrm{TSS})+(d+1)\ln n$ of BIC, which ranks models of equal size as BIC does (Proposition 5.14(i)) and here selects the same four predictors. The thresholds of the remark after Proposition 5.14 are $\sqrt2=1.414$ and $\sqrt{\ln n}=2.448$.

**Example 5.4** (Validation and cross-validation; ISLR §6.1.3, Figure 6.3). A validation subsample of one quarter of the observations, and ten-fold cross-validation, with the search of Definition 5.16 rerun inside each training set:

In [ ]:
pred_d <- function(fit, newdata, d) {
  cf <- coef(fit, id = d); drop(model.matrix(Balance ~ ., newdata)[, names(cf)] %*% cf) }
set.seed(1); tr_c <- sample(n_c, 3 * n_c / 4)
fit_tr <- regsubsets(Balance ~ ., data = Credit[tr_c, ], nvmax = 11)
val_c <- sapply(1:11, function(d) mean((yc[-tr_c] - pred_d(fit_tr, Credit[-tr_c, ], d))^2))
set.seed(1); folds_c <- sample(rep(1:10, length = n_c))
mse_c <- sapply(1:10, function(j) {
  fj <- regsubsets(Balance ~ ., data = Credit[folds_c != j, ], nvmax = 11)
  sapply(1:11, function(d) mean((yc[folds_c == j] - pred_d(fj, Credit[folds_c == j, ], d))^2)) })
cv_c <- rowMeans(mse_c); se_c <- apply(mse_c, 1, sd) / sqrt(10)
round(rbind(validation = val_c, CV = cv_c, SE = se_c))
c(validation_min = which.min(val_c), CV_min = which.min(cv_c),
  one_SE = min(which(cv_c <= min(cv_c) + se_c[which.min(cv_c)])))
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.8, 0.6, 0.4), mgp = c(2.2, 0.6, 0))
plot(d_c, sqrt(bic_c), type = "b", pch = 16, cex = 0.6, xlab = "number of predictors", ylab = "square root of BIC")
points(which.min(bic_c), sqrt(min(bic_c)), col = "blue", pch = 4, cex = 2, lwd = 2)
plot(d_c, sqrt(val_c), type = "b", pch = 16, cex = 0.6, xlab = "number of predictors", ylab = "validation error (root)")
points(which.min(val_c), sqrt(min(val_c)), col = "blue", pch = 4, cex = 2, lwd = 2)
plot(d_c, sqrt(cv_c), type = "b", pch = 16, cex = 0.6, xlab = "number of predictors", ylab = "CV error (root)")
points(which.min(cv_c), sqrt(min(cv_c)), col = "blue", pch = 4, cex = 2, lwd = 2)
par(op)

*Figure 5.3.* Square roots of BIC, of the validation error and of the ten-fold cross-validation error for the best `Credit` model of each size (ISLR Figure 6.3).

The validation subsample selects four predictors and cross-validation six, and the one-standard-error rule reduces six to four; ISLR's split and folds, which are not reproducible, gave six, six and three. From four predictors on, the cross-validation errors lie between $9944$ and $10{,}104$, with standard errors of $500$ to $574$; the three-predictor model's $10{,}802$ exceeds the one-standard-error bound $9944+500$.

## 3 Principal Components Analysis

> **Reading.** ISLR §12.2, §6.3.1 (overview of principal components analysis). SRM learning outcomes 5(a), 5(b).

### 3.1 Loadings, scores and variances

**Definition 5.19** (Principal components; ISLR §12.2.1, §6.3.1). Let $\mathbf{X}\neq\mathbf{0}$ be an $n\times p$ data matrix whose columns have been centred to mean zero, with rows $\mathbf{x}_1',\ldots,\mathbf{x}_n'$ and $\mathbf{S}=n^{-1}\mathbf{X}'\mathbf{X}$, the matrix of sample variances and covariances with divisor $n$. For a unit vector $\boldsymbol{\phi}\in\mathbb{R}^p$ the scores $\mathbf{z}=\mathbf{X}\boldsymbol{\phi}$ have mean zero and sample variance $n^{-1}\|\mathbf{X}\boldsymbol{\phi}\|^2=\boldsymbol{\phi}'\mathbf{S}\boldsymbol{\phi}$. The **first principal component loading vector** $\boldsymbol{\phi}_1$ maximises $\boldsymbol{\phi}'\mathbf{S}\boldsymbol{\phi}$ over unit vectors; for $m\geqslant2$, $\boldsymbol{\phi}_m$ maximises it over unit vectors whose scores are uncorrelated with $\mathbf{z}_1,\ldots,\mathbf{z}_{m-1}$. The entries $\phi_{jm}$ are the **loadings** and $\mathbf{z}_m=\mathbf{X}\boldsymbol{\phi}_m$, with entries $z_{im}=\sum_j\phi_{jm}x_{ij}$, is the $m$th **score vector**.

**Theorem 5.20** (Principal components are eigenvectors; ISLR §12.2.1). Let $\mathbf{S}=\mathbf{V}\boldsymbol{\Lambda}\mathbf{V}'$ with $\mathbf{V}=[\mathbf{v}_1\ \cdots\ \mathbf{v}_p]$ orthogonal and $\boldsymbol{\Lambda}=\operatorname{diag}(\lambda_1,\ldots,\lambda_p)$, $\lambda_1\geqslant\cdots\geqslant\lambda_p$, and let $r=\operatorname{rank}(\mathbf{X})$.

(i) $\lambda_p\geqslant0$, $r\leqslant\min(n-1,p)$, and $\lambda_m>0$ iff $m\leqslant r$.

(ii) For a unit $\boldsymbol{\phi}$ and an $l$ with $\lambda_l>0$, the scores $\mathbf{X}\boldsymbol{\phi}$ are uncorrelated with $\mathbf{X}\mathbf{v}_l$ iff $\boldsymbol{\phi}\perp\mathbf{v}_l$.

(iii) For $m\leqslant r$, $\boldsymbol{\phi}_m=\mathbf{v}_m$ solves the $m$th problem of Definition 5.19 when $\boldsymbol{\phi}_l=\mathbf{v}_l$ for $l<m$, and the maximum is $\lambda_m=n^{-1}\|\mathbf{z}_m\|^2$.

(iv) $\mathbf{z}_m'\mathbf{z}_l=n\lambda_m$ if $m=l$ and $0$ otherwise: the score vectors are orthogonal, hence uncorrelated.

(v) If $\lambda_1>\lambda_2>\cdots>\lambda_M>\lambda_{M+1}$ for some $M<r$, or $\lambda_1>\cdots>\lambda_r$ with $M=r$, then the first $M$ loading vectors are unique up to sign, and so are their score vectors.

**Proof.** **Step 1: rank and signs.** $\boldsymbol{\phi}'\mathbf{S}\boldsymbol{\phi}=n^{-1}\|\mathbf{X}\boldsymbol{\phi}\|^2\geqslant0$, so $\mathbf{S}$ is positive semi-definite and $\lambda_p\geqslant0$ by the prerequisites' proposition on the eigenvalues of positive semi-definite matrices. Every column of $\mathbf{X}$ is orthogonal to $\mathbf{1}$, so $\mathcal{C}(\mathbf{X})$ lies in the orthogonal complement of $\mathbf{1}$, of dimension $n-1$, and $r\leqslant\min(n-1,p)$. By the prerequisites' proposition **Gram matrices**, $\operatorname{rank}(\mathbf{S})=\operatorname{rank}(\mathbf{X}'\mathbf{X})=r$; as $\mathbf{V}$ is invertible, $\operatorname{rank}(\mathbf{S})=\operatorname{rank}(\boldsymbol{\Lambda})$, the number of nonzero $\lambda_m$. So exactly $\lambda_1,\ldots,\lambda_r$ are positive.

**Step 2: uncorrelated means orthogonal.** Both score vectors have mean zero, and their sample covariance is $n^{-1}(\mathbf{X}\boldsymbol{\phi})'(\mathbf{X}\mathbf{v}_l)=\boldsymbol{\phi}'\mathbf{S}\mathbf{v}_l=\lambda_l\boldsymbol{\phi}'\mathbf{v}_l$, which is zero iff $\boldsymbol{\phi}'\mathbf{v}_l=0$ when $\lambda_l>0$.

**Step 3: the maximum.** Write $\boldsymbol{\phi}=\sum_jc_j\mathbf{v}_j$ with $\sum_jc_j^2=1$. For $m\leqslant r$ the constraint of the $m$th problem, with $\boldsymbol{\phi}_l=\mathbf{v}_l$ and $\lambda_l>0$ for $l<m$, is $c_1=\cdots=c_{m-1}=0$ by Step 2, and then

$$
\boldsymbol{\phi}'\mathbf{S}\boldsymbol{\phi}=\sum_{j\geqslant m}\lambda_jc_j^2\leqslant\lambda_m\sum_{j\geqslant m}c_j^2
=\lambda_m ,
$$

with equality at $\boldsymbol{\phi}=\mathbf{v}_m$, which satisfies the constraint. The maximum is $\mathbf{v}_m'\mathbf{S}\mathbf{v}_m=\lambda_m=n^{-1}\|\mathbf{X}\mathbf{v}_m\|^2$.

**Step 4: the scores.** $\mathbf{z}_m'\mathbf{z}_l=\mathbf{v}_m'\mathbf{X}'\mathbf{X}\mathbf{v}_l=n\mathbf{v}_m'\mathbf{S}\mathbf{v}_l=n\lambda_l\mathbf{v}_m'\mathbf{v}_l$.

**Step 5: uniqueness.** Induct on $m\leqslant M$, assuming the first $m-1$ loadings are $\pm\mathbf{v}_1,\ldots,\pm\mathbf{v}_{m-1}$; for $m=1$ there is no constraint. The constraint is again $c_1=\cdots=c_{m-1}=0$, and equality in Step 3 requires $\sum_{j>m}(\lambda_m-\lambda_j)c_j^2=0$. Under the hypothesis $\lambda_j<\lambda_m$ for every $j>m$, so $c_j=0$ for $j>m$, and $\boldsymbol{\phi}=c_m\mathbf{v}_m$ with $c_m=\pm1$. The score vector is then $\pm\mathbf{X}\mathbf{v}_m$. ∎

**Remark** (ISLR §12.2.1, §12.2.4). The loading vectors are the eigenvectors of $\mathbf{X}'\mathbf{X}$, the component variances its eigenvalues divided by $n$, and at most $\min(n-1,p)$ components have positive variance. Flipping the sign of a loading vector and of its score vector changes neither the direction, nor the variance, nor the products $z_{im}\phi_{jm}$; different programs may return either sign. With tied eigenvalues the loadings are not unique even up to sign.

**Proposition 5.21** (Proportion of variance explained; ISLR §12.2.3). The **total variance** $\sum_jn^{-1}\sum_ix_{ij}^2$ equals $\operatorname{tr}(\mathbf{S})=\lambda_1+\cdots+\lambda_p$, and the **proportion of variance explained** by the $m$th component, $\mathbf{z}_m=\mathbf{X}\mathbf{v}_m$,

$$
\mathrm{PVE}_m=\frac{\sum_iz_{im}^2}{\sum_j\sum_ix_{ij}^2}=\frac{\lambda_m}{\lambda_1+\cdots+\lambda_p},
\qquad m=1,\ldots,p,
$$

is positive for $m\leqslant r$, zero for $m>r$, and nonincreasing in $m$, and $\mathrm{PVE}_1+\cdots+\mathrm{PVE}_r=1$. The cumulative $\mathrm{PVE}_1+\cdots+\mathrm{PVE}_M$ is nondecreasing in $M$ and reaches $1$ at $M=r$.

**Proof.** $\operatorname{tr}(\mathbf{S})=\sum_jn^{-1}\sum_ix_{ij}^2$ by definition of the trace, and $\operatorname{tr}(\mathbf{V}\boldsymbol{\Lambda}\mathbf{V}')=\operatorname{tr}(\boldsymbol{\Lambda}\mathbf{V}'\mathbf{V})=\operatorname{tr}(\boldsymbol{\Lambda})$ by the prerequisites' properties of the trace. $\sum_iz_{im}^2=n\lambda_m$ by Theorem 5.20(iv), positive for $m\leqslant r$ and zero for $m>r$ by (i), and $\lambda_m$ is nonincreasing in $m$; so the $\mathrm{PVE}_m$, $m\leqslant r$, sum to $1$. ∎

### 3.2 The closest subspaces

**Theorem 5.22** (Best approximation and closest subspaces; ISLR §12.2.2, §12.2.3). Let $1\leqslant M\leqslant p$, let $\boldsymbol{\Phi}_M=[\mathbf{v}_1\ \cdots\ \mathbf{v}_M]$ and $\mathbf{Z}_M=\mathbf{X}\boldsymbol{\Phi}_M$, and let $\|\cdot\|_F$ be the Frobenius norm, $\|\mathbf{A}\|_F^2=\sum_{i,j}a_{ij}^2=\operatorname{tr}(\mathbf{A}'\mathbf{A})$.

(i) For every $p\times M$ matrix $\mathbf{Q}$ with orthonormal columns, $\operatorname{tr}(\mathbf{Q}'\mathbf{S}\mathbf{Q})\leqslant\lambda_1+\cdots+\lambda_M$, with equality at $\mathbf{Q}=\boldsymbol{\Phi}_M$ and, if $M<p$ and $\lambda_M>\lambda_{M+1}$, only when $\mathcal{C}(\mathbf{Q})=\mathcal{C}(\boldsymbol{\Phi}_M)$.

(ii) For all $\mathbf{A}\in\mathbb{R}^{n\times M}$ and $\mathbf{B}\in\mathbb{R}^{p\times M}$,

$$
\sum_{i=1}^n\sum_{j=1}^p\Bigl(x_{ij}-\sum_{m=1}^Ma_{im}b_{jm}\Bigr)^2
      =\|\mathbf{X}-\mathbf{A}\mathbf{B}'\|_F^2\geqslant n\sum_{m>M}\lambda_m ,
$$

with equality at $\mathbf{A}=\mathbf{Z}_M$, $\mathbf{B}=\boldsymbol{\Phi}_M$, that is $a_{im}=z_{im}$ and $b_{jm}=\phi_{jm}$. For $M\geqslant r$ the approximation is exact: $\mathbf{X}=\mathbf{Z}_M\boldsymbol{\Phi}_M'$.

(iii) $n^{-1}\|\mathbf{X}\|_F^2=\sum_{m\leqslant M}n^{-1}\|\mathbf{z}_m\|^2+n^{-1}\|\mathbf{X}-\mathbf{Z}_M\boldsymbol{\Phi}_M'\|_F^2$, so $\mathrm{PVE}_1+\cdots+\mathrm{PVE}_M=1-\mathrm{RSS}/\mathrm{TSS}$ with $\mathrm{RSS}=\|\mathbf{X}-\mathbf{Z}_M\boldsymbol{\Phi}_M'\|_F^2$ and $\mathrm{TSS}=\|\mathbf{X}\|_F^2$.

(iv) Let $\tilde{\mathbf{x}}_i=\mathbf{x}_i+\bar{\mathbf{x}}$ be the uncentred observations. Over all affine subspaces $\mathbf{a}+W$ of $\mathbb{R}^p$ with $\dim W=M$, the sum of squared distances $\sum_i\mathrm{dist}(\tilde{\mathbf{x}}_i,\mathbf{a}+W)^2$ is smallest, equal to $n\sum_{m>M}\lambda_m$, for $\bar{\mathbf{x}}+\mathcal{C}(\boldsymbol{\Phi}_M)$, and, if $M=p$ or $\lambda_M>\lambda_{M+1}$, for no other. For $M=1$: the line through $\bar{\mathbf{x}}$ along $\boldsymbol{\phi}_1$ is a line closest to the observations, and, if $p=1$ or $\lambda_1>\lambda_2$, the only one.

**Proof.** **Step 1: (i).** Write $\mathbf{S}=\sum_j\lambda_j\mathbf{v}_j\mathbf{v}_j'$ and $\omega_j=\|\mathbf{Q}'\mathbf{v}_j\|^2$. Then $\operatorname{tr}(\mathbf{Q}'\mathbf{S}\mathbf{Q})=\sum_j\lambda_j\operatorname{tr}(\mathbf{Q}'\mathbf{v}_j\mathbf{v}_j'\mathbf{Q})=\sum_j\lambda_j\omega_j$. The matrix $\mathbf{Q}\mathbf{Q}'$ is symmetric and idempotent, so $\omega_j=\mathbf{v}_j'\mathbf{Q}\mathbf{Q}'\mathbf{v}_j=\|\mathbf{Q}\mathbf{Q}'\mathbf{v}_j\|^2$ lies in $[0,1]$ by Pythagoras, $\mathbf{v}_j$ being a unit vector; and $\sum_j\omega_j=\operatorname{tr}(\mathbf{Q}'\mathbf{V}\mathbf{V}'\mathbf{Q})=\operatorname{tr}(\mathbf{Q}'\mathbf{Q})=M$. Therefore

$$
\begin{align*}
\sum_j\lambda_j\omega_j-\sum_{j\leqslant M}\lambda_j
&=\sum_{j\leqslant M}\lambda_j(\omega_j-1)+\sum_{j>M}\lambda_j\omega_j\\
&\leqslant\lambda_M\sum_{j\leqslant M}(\omega_j-1)+\lambda_M\sum_{j>M}\omega_j
=\lambda_M\Bigl(\sum_j\omega_j-M\Bigr)=0,
\end{align*}
$$

using $\omega_j-1\leqslant0$ with $\lambda_j\geqslant\lambda_M$ for $j\leqslant M$, and $\omega_j\geqslant0$ with $\lambda_j\leqslant\lambda_M$ for $j>M$. At $\mathbf{Q}=\boldsymbol{\Phi}_M$, $\mathbf{Q}'\mathbf{S}\mathbf{Q}=\operatorname{diag}(\lambda_1,\ldots,\lambda_M)$. The right side minus the left side of the inequality is $\sum_{j\leqslant M}(\lambda_M-\lambda_j)(\omega_j-1)+\sum_{j>M}(\lambda_M-\lambda_j)\omega_j$, a sum of nonnegative terms. If $M<p$ and $\lambda_M>\lambda_{M+1}$, equality forces $\omega_j=0$ for $j>M$, where $\lambda_M-\lambda_j\geqslant\lambda_M-\lambda_{M+1}>0$; then $\sum_{j\leqslant M}\omega_j=M$ with every $\omega_j\leqslant1$ forces $\omega_j=1$, that is $\|\mathbf{Q}\mathbf{Q}'\mathbf{v}_j\|=\|\mathbf{v}_j\|$, for $j\leqslant M$. By Pythagoras $(\mathbf{I}-\mathbf{Q}\mathbf{Q}')\mathbf{v}_j=\mathbf{0}$, so $\mathbf{v}_1,\ldots,\mathbf{v}_M\in\mathcal{C}(\mathbf{Q})$, and $\mathcal{C}(\mathbf{Q})=\mathcal{C}(\boldsymbol{\Phi}_M)$, both having dimension $M$.

**Step 2: a projection bound.** Let $W$ be a subspace of $\mathbb{R}^p$ with $\dim W\leqslant M$ and projection matrix $\mathbf{P}_W$. Complete an orthonormal basis of $W$, the columns of $\mathbf{Q}_1$, to $M$ orthonormal vectors $\mathbf{Q}=[\mathbf{Q}_1\ \mathbf{Q}_2]$. By the prerequisites' theorem **orthogonal projection**, $\mathbf{P}_W=\mathbf{Q}_1\mathbf{Q}_1'$, so $\mathbf{Q}\mathbf{Q}'-\mathbf{P}_W=\mathbf{Q}_2\mathbf{Q}_2'$ and $\operatorname{tr}\{(\mathbf{Q}\mathbf{Q}'-\mathbf{P}_W)\mathbf{S}\}=\operatorname{tr}(\mathbf{Q}_2'\mathbf{S}\mathbf{Q}_2)\geqslant0$, $\mathbf{S}$ being positive semi-definite. With Step 1,

$$
\begin{align*}
\sum_i\|(\mathbf{I}-\mathbf{P}_W)\mathbf{x}_i\|^2&=\operatorname{tr}\{\mathbf{X}(\mathbf{I}-\mathbf{P}_W)\mathbf{X}'\}=n\operatorname{tr}(\mathbf{S})-n\operatorname{tr}(\mathbf{P}_W\mathbf{S})\\
&\geqslant n\sum_j\lambda_j-n\operatorname{tr}(\mathbf{Q}'\mathbf{S}\mathbf{Q})\geqslant n\sum_{m>M}\lambda_m .
\end{align*}
$$

**Step 3: (ii).** Row $i$ of $\mathbf{A}\mathbf{B}'$ is $(\mathbf{B}\mathbf{a}_i)'$, $\mathbf{a}_i'$ being row $i$ of $\mathbf{A}$, and $\mathbf{B}\mathbf{a}_i\in W=\mathcal{C}(\mathbf{B})$, of dimension at most $M$. By the prerequisites' theorem **orthogonal projection**, $\|\mathbf{x}_i-\mathbf{B}\mathbf{a}_i\|\geqslant\|(\mathbf{I}-\mathbf{P}_W)\mathbf{x}_i\|$, and summing over $i$ with Step 2 gives the bound. At $\mathbf{A}=\mathbf{Z}_M$, $\mathbf{B}=\boldsymbol{\Phi}_M$, $\mathbf{X}-\mathbf{Z}_M\boldsymbol{\Phi}_M'=\mathbf{X}(\mathbf{I}-\boldsymbol{\Phi}_M\boldsymbol{\Phi}_M')$, and since $\mathbf{I}-\boldsymbol{\Phi}_M\boldsymbol{\Phi}_M'$ is symmetric and idempotent, $\|\mathbf{X}(\mathbf{I}-\boldsymbol{\Phi}_M\boldsymbol{\Phi}_M')\|_F^2=n\operatorname{tr}\{(\mathbf{I}-\boldsymbol{\Phi}_M\boldsymbol{\Phi}_M')\mathbf{S}\} =n\bigl(\sum_j\lambda_j-\sum_{m\leqslant M}\lambda_m\bigr)$. For $M\geqslant r$ this is $0$ by Theorem 5.20(i).

**Step 4: (iii).** $\|\mathbf{X}\|_F^2=n\operatorname{tr}(\mathbf{S})=n\sum_j\lambda_j$ and $\sum_{m\leqslant M}\|\mathbf{z}_m\|^2=n\sum_{m\leqslant M}\lambda_m$ by Theorem 5.20(iv); Step 3 gives the remainder, and dividing by $\|\mathbf{X}\|_F^2$ gives the $\mathrm{PVE}$ form by Proposition 5.21.

**Step 5: (iv).** By the prerequisites' theorem **orthogonal projection** applied to $\tilde{\mathbf{x}}_i-\mathbf{a}$ and $W$, $\mathrm{dist}(\tilde{\mathbf{x}}_i,\mathbf{a}+W)=\|(\mathbf{I}-\mathbf{P}_W)(\tilde{\mathbf{x}}_i-\mathbf{a})\|$. With $\tilde{\mathbf{x}}_i-\mathbf{a}=\mathbf{x}_i+(\bar{\mathbf{x}}-\mathbf{a})$ and $\sum_i\mathbf{x}_i=\mathbf{0}$ the cross terms sum to zero, and

$$
\sum_i\mathrm{dist}(\tilde{\mathbf{x}}_i,\mathbf{a}+W)^2=\sum_i\|(\mathbf{I}-\mathbf{P}_W)\mathbf{x}_i\|^2+n\|(\mathbf{I}-\mathbf{P}_W)(\bar{\mathbf{x}}-\mathbf{a})\|^2
\geqslant n\sum_{m>M}\lambda_m
$$

by Step 2, with equality at $\mathbf{a}=\bar{\mathbf{x}}$ and $W=\mathcal{C}(\boldsymbol{\Phi}_M)$ by Step 3. If $\dim W=M$ and the minimum is attained, then $(\mathbf{I}-\mathbf{P}_W)(\bar{\mathbf{x}}-\mathbf{a})=\mathbf{0}$ and equality holds in Step 2, where $\mathbf{Q}=\mathbf{Q}_1$, so $\operatorname{tr}(\mathbf{Q}_1'\mathbf{S}\mathbf{Q}_1)=\lambda_1+\cdots+\lambda_M$. If $M=p$, $W=\mathbb{R}^p$; if $\lambda_M>\lambda_{M+1}$, Step 1 gives $W=\mathcal{C}(\mathbf{Q}_1)=\mathcal{C}(\boldsymbol{\Phi}_M)$. In both cases $\bar{\mathbf{x}}-\mathbf{a}\in W$ gives $\mathbf{a}+W=\bar{\mathbf{x}}+\mathcal{C}(\boldsymbol{\Phi}_M)$. ∎

**Remark** (ISLR §12.2.3). The cumulative proportion of variance explained is an $R^2$: the first $M$ score and loading vectors give the approximation $\hat{\mathbf{X}}=\mathbf{Z}_M\boldsymbol{\Phi}_M'$ of the centred data in Theorem 5.22, and $\mathrm{PVE}_1+\cdots+\mathrm{PVE}_M =1-\sum_{i,j}(x_{ij}-\hat x_{ij})^2/\sum_{i,j}x_{ij}^2$, the proportion of the total sum of squares that the approximation reproduces; this is part (iii) of Theorem 5.22 divided through by $n^{-1}\|\mathbf{X}\|_F^2$.

**Proposition 5.23** (Principal components through the singular value decomposition; ISLR §12.2.1). Let $\mathbf{X}=\mathbf{U}\mathbf{D}\mathbf{V}_r'$ be the singular value decomposition of the prerequisites, with $\mathbf{U}=[\mathbf{u}_1\ \cdots\ \mathbf{u}_r]$, $\mathbf{V}_r=[\mathbf{w}_1\ \cdots\ \mathbf{w}_r]$ and $d_1\geqslant\cdots\geqslant d_r>0$. Then $\mathbf{S}\mathbf{w}_m=(d_m^2/n)\mathbf{w}_m$ and $d_m^2/n=\lambda_m$, so $\mathbf{w}_1,\ldots,\mathbf{w}_r$ may serve as $\mathbf{v}_1,\ldots,\mathbf{v}_r$ in Theorem 5.20, and then $\mathbf{z}_m=d_m\mathbf{u}_m$. The standard deviations $d_m/\sqrt{n-1}$ reported by `prcomp`, which divides by $n-1$, give the same $\mathrm{PVE}_m$.

**Proof.** $\mathbf{X}'\mathbf{X}=\mathbf{V}_r\mathbf{D}\mathbf{U}'\mathbf{U}\mathbf{D}\mathbf{V}_r'=\mathbf{V}_r\mathbf{D}^2\mathbf{V}_r'$, so $\mathbf{X}'\mathbf{X}\mathbf{w}_m=d_m^2\mathbf{w}_m$ by orthonormality of the $\mathbf{w}_m$, and the $d_m^2$ are the nonzero eigenvalues of $\mathbf{X}'\mathbf{X}=n\mathbf{S}$ in decreasing order by the prerequisites' theorem **SVD**. Finally $\mathbf{X}\mathbf{w}_m=\mathbf{U}\mathbf{D}\mathbf{V}_r'\mathbf{w}_m=d_m\mathbf{u}_m$, and $\mathrm{PVE}_m=d_m^2/\sum_ld_l^2$ whatever the common divisor. ∎

**Proposition 5.24** (Scaling changes the components; ISLR §12.2.4). Let $\mathbf{D}_c=\operatorname{diag}(c,1,\ldots,1)$, $c>0$, rescale the first variable, so that the data become $\mathbf{X}\mathbf{D}_c$ and $\mathbf{S}$ becomes $\mathbf{S}(c)=\mathbf{D}_c\mathbf{S}\mathbf{D}_c$, and let $s_{11}>0$.

(i) The largest eigenvalue of any $\mathbf{S}$ is at least $\max_js_{jj}$.

(ii) As $c\to\infty$, every unit leading eigenvector $\boldsymbol{\phi}(c)$ of $\mathbf{S}(c)$ has $\phi_1(c)^2\to1$, and $\mathrm{PVE}_1\to1$.

Standardising each variable to unit variance replaces $\mathbf{S}$ by a multiple of the correlation matrix, which does not depend on the units of measurement.

**Proof.** (i) By the prerequisites' corollary **Rayleigh**, $\lambda_1\geqslant\mathbf e_j'\mathbf{S}\mathbf e_j=s_{jj}$.

(ii) Write $\boldsymbol{\phi}=\boldsymbol{\phi}(c)=(\phi_1,\boldsymbol{\phi}_2')'$ and partition $\mathbf{S}$ with $s_{11}$, the vector $\mathbf{s}_{21}$ and the block $\mathbf{S}_{22}$. The first diagonal entry of $\mathbf{S}(c)$ is $c^2s_{11}$, so by (i)

$$
c^2s_{11}\leqslant\boldsymbol{\phi}'\mathbf{S}(c)\boldsymbol{\phi}=c^2s_{11}\phi_1^2+2c\phi_1\mathbf{s}_{21}'\boldsymbol{\phi}_2+\boldsymbol{\phi}_2'\mathbf{S}_{22}\boldsymbol{\phi}_2
\leqslant c^2s_{11}\phi_1^2+2c\|\mathbf{s}_{21}\|+\operatorname{tr}(\mathbf{S}_{22}),
$$

using the Cauchy–Schwarz inequality, $|\phi_1|\leqslant1$, $\|\boldsymbol{\phi}_2\|\leqslant1$ and the prerequisites' corollary **Rayleigh** for $\mathbf{S}_{22}$, whose largest eigenvalue is at most its trace: $\mathbf{v}'\mathbf{S}_{22}\mathbf{v}=(0,\mathbf{v}')\mathbf{S}(0,\mathbf{v}')'\geqslant0$, so $\mathbf{S}_{22}$ is positive semi-definite and its eigenvalues are nonnegative by the prerequisites' proposition on positive semi-definite matrices, and they sum to its trace as in the proof of Proposition 5.21. Hence $s_{11}(1-\phi_1^2)\leqslant\{2c\|\mathbf{s}_{21}\|+\operatorname{tr}(\mathbf{S}_{22})\}/c^2\to0$. Also $\mathrm{PVE}_1\geqslant c^2s_{11}/\{c^2s_{11}+\operatorname{tr}(\mathbf{S}_{22})\}\to1$. A standardised variable $x_{ij}/s_j$ is unchanged when column $j$ is multiplied by a positive constant, and the matrix $\mathbf{S}$ of the standardised data is the correlation matrix times $(n-1)/n$ when $s_j$ has divisor $n-1$. ∎

**Remark** (ISLR §12.2.4, §12.2.5). In least squares regression a rescaled column rescales its coefficient and changes no fitted value (Unit 3's proposition **codings are reparametrisations**); principal components have no such invariance, which is why variables measured in different units are standardised first. The number of components used for visualisation is usually read from a **scree plot** of $\mathrm{PVE}_m$ against $m$, at an elbow after which the $\mathrm{PVE}_m$ are small; the choice is **ad hoc**. When the scores serve a supervised method, such as a regression on the first $M$ score vectors, $M$ is a tuning parameter chosen by cross-validation. Scores also serve as inputs to classification and clustering, and as a two-dimensional view of the observations.

**Example 5.5** (`USArrests`; ISLR §12.2, Table 12.1, Figures 12.1, 12.3 and 12.4). Arrests per $100{,}000$ residents for `Murder`, `Assault` and `Rape`, and the percentage urban `UrbanPop`, for the $50$ states, standardised; signs are set as in ISLR Table 12.1.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.1)
pr <- prcomp(USArrests, scale = TRUE)
s_islr <- c(sign(pr$rotation["Murder", 1]), sign(pr$rotation["UrbanPop", 2]), 1, 1)
pr$rotation <- sweep(pr$rotation, 2, s_islr, "*"); pr$x <- sweep(pr$x, 2, s_islr, "*")
round(pr$rotation, 4)
round(rbind(sdev = pr$sdev, PVE = pr$sdev^2 / sum(pr$sdev^2),
            cumulative = cumsum(pr$sdev^2) / sum(pr$sdev^2)), 4)
round(apply(USArrests, 2, var), 2)
pr_u <- prcomp(USArrests, scale = FALSE)
round(pr_u$rotation[, 1:2], 4)
round(pr_u$sdev^2 / sum(pr_u$sdev^2), 4)
rbind(PC1_high = names(sort(pr$x[, 1], decreasing = TRUE))[1:3],
      PC1_low = names(sort(pr$x[, 1]))[1:3], PC2_high = names(sort(pr$x[, 2], decreasing = TRUE))[1:3],
      PC2_low = names(sort(pr$x[, 2]))[1:3])
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.2, 3, 3), mgp = c(2, 0.6, 0), cex = 0.6)
biplot(pr, scale = 0, cex = 0.7, xlabs = abbreviate(rownames(USArrests), 4))
biplot(pr_u, scale = 0, cex = 0.7, xlabs = abbreviate(rownames(USArrests), 4))
par(op)

*Figure 5.4.* Biplots of the first two principal components of `USArrests`: state scores (abbreviated names, bottom and left axes) and loading vectors (arrows, top and right axes). Left, standardised variables (ISLR Figure 12.1); right, unscaled (ISLR Figure 12.4).

The first loading vector puts nearly equal weights, $0.5359$, $0.5832$ and $0.5434$, on the three crime rates and $0.2782$ on `UrbanPop`: it measures overall serious crime, highest in Florida, Nevada and California and lowest in North Dakota, Vermont and Maine. The second puts $0.8728$ on `UrbanPop`: it measures urbanisation, highest in Hawaii, California and Rhode Island and lowest in Mississippi and the Carolinas. The first two components explain proportions $0.6201$ and $0.2474$ of the variance, $0.8675$ together. Unscaled, `Assault`, with variance $6945.17$ against at most $209.52$ for the others, receives the first loading $0.9952$, and the first component explains a proportion $0.9655$ of the variance, as Proposition 5.24 predicts.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
pve_us <- pr$sdev^2 / sum(pr$sdev^2)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.2, 0.6, 0))
plot(pve_us, type = "b", ylim = c(0, 1), xlab = "principal component", ylab = "PVE", pch = 16, xaxt = "n")
axis(1, at = 1:4)
plot(cumsum(pve_us), type = "b", ylim = c(0, 1), xlab = "principal component", ylab = "cumulative PVE",
     pch = 16, xaxt = "n")
axis(1, at = 1:4)
par(op)

*Figure 5.5.* Scree plot and cumulative proportion of variance explained for the standardised `USArrests` (ISLR Figure 12.3).

The scree plot has its elbow after the second component: the third and fourth explain $0.0891$ and $0.0434$.

## 4 PCR and Partial Least Squares

> **Reading.** ISLR §6.3. SRM learning outcomes 2(e), 2(g).

### 4.1 Dimension reduction

**Definition 5.25** (Dimension reduction; ISLR §6.3). Let $\mathbf{X}$ be the $n\times p$ matrix of the predictors with centred columns, as in Definition 5.19, and $\mathbf{Y}$ the response. For a $p\times M$ matrix $\boldsymbol{\Phi}$ with $M\leqslant p$, the **derived predictors** are the columns of $\mathbf{Z}=\mathbf{X}\boldsymbol{\Phi}$, $z_{im}=\sum_j\phi_{jm}x_{ij}$, and a **dimension reduction** fit is the least squares fit of $y_i=\theta_0+\sum_m\theta_mz_{im}+\varepsilon_i$, with coefficients $\hat\theta_0,\hat{\boldsymbol{\theta}}$ and implied slopes $\hat{\boldsymbol{\beta}}=\boldsymbol{\Phi}\hat{\boldsymbol{\theta}}$, $\hat\beta_j=\sum_m\hat\theta_m\phi_{jm}$.

**Proposition 5.26** (Dimension reduction constrains the slopes; ISLR §6.3). Let $[\mathbf{1}\ \mathbf{Z}]$ have full column rank.

(i) The dimension reduction fit is the least squares fit of $\mathbf{Y}$ on $[\mathbf{1}\ \mathbf{X}]$ subject to $\boldsymbol{\beta}\in\mathcal{C}(\boldsymbol{\Phi})$, and its slopes are $\hat{\boldsymbol{\beta}}=\boldsymbol{\Phi}\hat{\boldsymbol{\theta}}$.

(ii) If $M=p$, $\boldsymbol{\Phi}$ is invertible and $\operatorname{rank}(\mathbf{X})=p$, it is the unconstrained least squares fit, with the least squares slopes.

**Proof.** (i) $\theta_0\mathbf{1}+\mathbf{Z}\boldsymbol{\theta}=\theta_0\mathbf{1}+\mathbf{X}(\boldsymbol{\Phi}\boldsymbol{\theta})$, so the mean vectors the two problems allow, $\{\theta_0\mathbf{1}+\mathbf{Z}\boldsymbol{\theta}\}$ and $\{\beta_0\mathbf{1}+\mathbf{X}\boldsymbol{\beta}:\boldsymbol{\beta}\in\mathcal{C}(\boldsymbol{\Phi})\}$, are the same set, and both minimise the same residual sum of squares over it; the minimiser is unique by Unit 2's theorem **the least squares estimator**, and $\boldsymbol{\beta}=\boldsymbol{\Phi}\boldsymbol{\theta}$ at the solution.

(ii) $[\mathbf{1}\ \mathbf{Z}]=[\mathbf{1}\ \mathbf{X}]\mathbf{A}$ with $\mathbf{A}=\operatorname{diag}(1,\boldsymbol{\Phi})$ invertible, and Unit 3's proposition **codings are reparametrisations** applies. ∎

**Definition 5.27** (Principal components regression; ISLR §6.3.1). **Principal components regression** with $M\leqslant r$ components, PCR($M$), is the dimension reduction fit with $\boldsymbol{\Phi}=\boldsymbol{\Phi}_M=[\mathbf{v}_1\ \cdots\ \mathbf{v}_M]$, the first $M$ loading vectors of $\mathbf{X}$, whose columns are usually standardised first.

**Theorem 5.28** (The PCR estimator; ISLR §6.3.1). Let $\operatorname{rank}(\mathbf{X})=p$, let $\mathbf{X}=\mathbf{U}\mathbf{D}\mathbf{V}'$ be its singular value decomposition with the $p\times p$ orthogonal matrix $\mathbf{V}=[\mathbf{v}_1\ \cdots\ \mathbf{v}_p]$ as in Proposition 5.23, and let $M\leqslant p$.

(i) $\hat\theta_0=\bar y$ and $\hat\theta_m=\mathbf{z}_m'\mathbf{Y}/\|\mathbf{z}_m\|^2=\mathbf{u}_m'\mathbf{Y}/d_m$, the slope of the simple regression of $y$ on $z_m$; it does not depend on $M$.

(ii) $\hat{\boldsymbol{\beta}}(M)=\sum_{m\leqslant M}\mathbf{v}_m\mathbf{u}_m'\mathbf{Y}/d_m$, the fitted values are $\bar y\mathbf{1}+\sum_{m\leqslant M}\mathbf{u}_m\mathbf{u}_m'\mathbf{Y}$, and $R^2(M)=\sum_{m\leqslant M}(\mathbf{u}_m'\mathbf{Y})^2/\mathrm{TSS}$: each component adds its own share.

(iii) Under A1–A3 with $\operatorname{\mathsf{E}}[\mathbf{Y}]=\beta_0\mathbf{1}+\mathbf{X}\boldsymbol{\beta}$,

$$
\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}(M)]=\sum_{m\leqslant M}\mathbf{v}_m\mathbf{v}_m'\boldsymbol{\beta},\qquad
      \operatorname{var}(\hat{\boldsymbol{\beta}}(M))=\sigma^2\sum_{m\leqslant M}\frac{\mathbf{v}_m\mathbf{v}_m'}{d_m^2}.
$$

(iv) $\operatorname{tr}\operatorname{var}(\hat{\boldsymbol{\beta}}(M))=\sigma^2\sum_{m\leqslant M}d_m^{-2}$ increases with $M$, and $\|\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}(M)]-\boldsymbol{\beta}\|^2=\sum_{m>M}(\mathbf{v}_m'\boldsymbol{\beta})^2$ does not.

(v) $\hat{\boldsymbol{\beta}}(p)=(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{Y}$, the least squares slopes, and PCR($p$) is the least squares fit.

**Proof.** **Step 1: an orthogonal design.** By Proposition 5.23, $\mathbf{z}_m=d_m\mathbf{u}_m$, and $\mathbf{1}'\mathbf{z}_m=0$ because the columns of $\mathbf{X}$ are centred. So the columns of $[\mathbf{1}\ \mathbf{Z}_M]$ are orthogonal, with squared lengths $n,d_1^2,\ldots,d_M^2$; the normal equations are diagonal and give $\hat\theta_0=\mathbf{1}'\mathbf{Y}/n$ and $\hat\theta_m=\mathbf{z}_m'\mathbf{Y}/d_m^2=\mathbf{u}_m'\mathbf{Y}/d_m$, whatever the other columns. As $\bar z_m=0$, the simple regression slope $\sum_i(z_{im}-\bar z_m)(y_i-\bar y)/\sum_i(z_{im}-\bar z_m)^2$ is $\mathbf{z}_m'\mathbf{Y}/\|\mathbf{z}_m\|^2$.

**Step 2: slopes and fit.** $\hat{\boldsymbol{\beta}}(M)=\boldsymbol{\Phi}_M\hat{\boldsymbol{\theta}}$ by Proposition 5.26(i), and the fitted values are $\hat\theta_0\mathbf{1}+\sum_m\hat\theta_m\mathbf{z}_m=\bar y\mathbf{1}+\sum_m\mathbf{u}_m\mathbf{u}_m'\mathbf{Y}$. The regression sum of squares is $\|\sum_m\mathbf{u}_m\mathbf{u}_m'\mathbf{Y}\|^2=\sum_m(\mathbf{u}_m'\mathbf{Y})^2$, the $\mathbf{u}_m$ being orthonormal, and $R^2=\mathrm{SSR}/\mathrm{SST}$ by Unit 2's theorem **variance decomposition**.

**Step 3: moments.** $\mathbf{u}_m'\mathbf{1}=0$ and $\mathbf{u}_m'\mathbf{X}=\mathbf{u}_m'\mathbf{U}\mathbf{D}\mathbf{V}'=d_m\mathbf{v}_m'$, so $\operatorname{\mathsf{E}}[\mathbf{u}_m'\mathbf{Y}]=d_m\mathbf{v}_m'\boldsymbol{\beta}$ and $\operatorname{\mathsf{E}}[\hat\theta_m]=\mathbf{v}_m'\boldsymbol{\beta}$. With $\mathbf{A}=\sum_{m\leqslant M}\mathbf{v}_m\mathbf{u}_m'/d_m$, $\hat{\boldsymbol{\beta}}(M)=\mathbf{A}\mathbf{Y}$ and, by the prerequisites' variance under a linear map, $\operatorname{var}(\hat{\boldsymbol{\beta}}(M))=\sigma^2\mathbf{A}\mathbf{A}' =\sigma^2\sum_{m,l}\mathbf{v}_m\mathbf{u}_m'\mathbf{u}_l\mathbf{v}_l'/(d_md_l)=\sigma^2\sum_m\mathbf{v}_m\mathbf{v}_m'/d_m^2$.

**Step 4: variance and bias.** $\operatorname{tr}(\mathbf{v}_m\mathbf{v}_m')=\|\mathbf{v}_m\|^2=1$. As $\mathbf{V}$ is orthogonal, $\boldsymbol{\beta}=\sum_{m\leqslant p}\mathbf{v}_m\mathbf{v}_m'\boldsymbol{\beta}$, so $\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}(M)]-\boldsymbol{\beta}=-\sum_{m>M}\mathbf{v}_m(\mathbf{v}_m'\boldsymbol{\beta})$, whose squared length is $\sum_{m>M}(\mathbf{v}_m'\boldsymbol{\beta})^2$.

**Step 5: $M=p$.** $(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'=(\mathbf{V}\mathbf{D}^2\mathbf{V}')^{-1}\mathbf{V}\mathbf{D}\mathbf{U}'=\mathbf{V}\mathbf{D}^{-1}\mathbf{U}'$, which is $\hat{\boldsymbol{\beta}}(p)$. Since $\mathbf{1}'\mathbf{X}=\mathbf{0}'$, the normal equations of $[\mathbf{1}\ \mathbf{X}]$ split into $n\hat\beta_0=\mathbf{1}'\mathbf{Y}$ and $\mathbf{X}'\mathbf{X}\hat{\boldsymbol{\beta}}=\mathbf{X}'\mathbf{Y}$, so these are the least squares slopes; Proposition 5.26(ii) gives the fit. ∎

**Remark** (ISLR §6.3.1). The components dropped first have the smallest $d_m$, and each contributes $\sigma^2/d_m^2$ to the total variance of $\hat{\boldsymbol{\beta}}$ and $(\mathbf{v}_m'\boldsymbol{\beta})^2$ to its squared bias: PCR trades the two along $M$, and it works when $\boldsymbol{\beta}$ lies close to the span of the first loading vectors, that is when the directions in which the predictors vary most are those related to the response. Nothing guarantees this, since the loadings are computed without the response. Every $\hat\beta_j$ generally involves all the components, so PCR is not a selection method. The predictors are standardised unless they share units, by Proposition 5.24.

### 4.2 Partial least squares

**Definition 5.29** (Partial least squares; ISLR §6.3.2). Standardise the predictors, and let $\mathbf{X}^{(1)}$ be the standardised matrix, whose columns have mean zero and sample variance one with divisor $n-1$. For $m=1,2,\ldots$ put

$$
\boldsymbol{\phi}_m=\mathbf{X}^{(m)\prime}\mathbf{Y},\qquad \mathbf{z}_m=\mathbf{X}^{(m)}\boldsymbol{\phi}_m,\qquad
\mathbf{X}^{(m+1)}=\mathbf{X}^{(m)}-\frac{\mathbf{z}_m\mathbf{z}_m'\mathbf{X}^{(m)}}{\mathbf{z}_m'\mathbf{z}_m},
$$

stopping if $\mathbf{z}_m=\mathbf{0}$: each column of $\mathbf{X}^{(m+1)}$ is the residual of the regression of the column of $\mathbf{X}^{(m)}$ on $\mathbf{z}_m$. PLS($M$) is the least squares fit of $\mathbf{Y}$ on $\mathbf{1},\mathbf{z}_1,\ldots,\mathbf{z}_M$. ISLR describes $\boldsymbol{\phi}_1$ through the slopes of the simple regressions of $y$ on each $x_j$, which are proportional to $\mathbf{X}^{(1)\prime}\mathbf{Y}$. For $m\geqslant2$ the inner products $\mathbf{X}^{(m)\prime}\mathbf{Y}$ are the rule of `plsr`; ISLR's “same fashion” would divide the $j$th of them by $\|\mathbf{x}_j^{(m)}\|^2$, which differs across $j$.

**Proposition 5.30** (Partial least squares; ISLR §6.3.2). Let $\mathcal{C}$ denote the column space of $\mathbf{X}=\mathbf{X}^{(1)}$.

(i) The slope of the simple regression of $y$ on $x_j$ is $\mathbf{x}_j'\mathbf{Y}/(n-1)$, which equals $r_js_y$ when $\mathrm{TSS}>0$, $r_j$ being the sample correlation of $x_j$ and $y$; and, if $\mathbf{X}'\mathbf{Y}\neq\mathbf{0}$, $\boldsymbol{\phi}_1/\|\boldsymbol{\phi}_1\|$ maximises $(\mathbf{Y}'\mathbf{X}\boldsymbol{\phi})^2$, the squared sample covariance of $y$ with $\mathbf{X}\boldsymbol{\phi}$ up to the factor $n^2$ or $(n-1)^2$, over unit $\boldsymbol{\phi}$.

(ii) The $\mathbf{z}_m$ are orthogonal to $\mathbf{1}$ and to each other, lie in $\mathcal{C}$, and $\mathbf{X}^{(m+1)}=(\mathbf{I}-\mathbf{P}_m)\mathbf{X}$, with $\mathbf{P}_m$ the projection onto $\mathcal{C}(\mathbf{z}_1,\ldots,\mathbf{z}_m)$.

(iii) The fitted values of PLS($M$) are $\bar y\mathbf{1}+\sum_{m\leqslant M}\mathbf{z}_m\mathbf{z}_m'\mathbf{Y}/\|\mathbf{z}_m\|^2$.

(iv) If $\mathbf{z}_m=\mathbf{0}$, then PLS($m-1$) is the least squares fit on $[\mathbf{1}\ \mathbf{X}]$; if $\mathbf{z}_1,\ldots,\mathbf{z}_M$ are nonzero with $M=\operatorname{rank}(\mathbf{X})$, so is PLS($M$). The least squares fit is reached in at most $\operatorname{rank}(\mathbf{X})$ steps.

**Proof.** (i) With $\bar x_j=0$ and $\sum_ix_{ij}^2=n-1$, the slope is $\mathbf{x}_j'(\mathbf{Y}-\bar y\mathbf{1})/\mathbf{x}_j'\mathbf{x}_j=\mathbf{x}_j'\mathbf{Y}/(n-1)$, and $r_j=\mathbf{x}_j'\mathbf{Y}/\{\sqrt{n-1}\,\|\mathbf{Y}-\bar y\mathbf{1}\|\}$ with $s_y=\|\mathbf{Y}-\bar y\mathbf{1}\|/\sqrt{n-1}$. By the Cauchy–Schwarz inequality, $(\mathbf{Y}'\mathbf{X}\boldsymbol{\phi})^2=\{(\mathbf{X}'\mathbf{Y})'\boldsymbol{\phi}\}^2\leqslant\|\mathbf{X}'\mathbf{Y}\|^2$ for unit $\boldsymbol{\phi}$, with equality iff $\boldsymbol{\phi}$ is proportional to $\mathbf{X}'\mathbf{Y}=\boldsymbol{\phi}_1$; and $\mathbf{Y}'\mathbf{X}\boldsymbol{\phi}=(\mathbf{Y}-\bar y\mathbf{1})'\mathbf{X}\boldsymbol{\phi}$ is $n$, or $n-1$, times the sample covariance.

(ii) Induct on $m$, with $\mathbf{P}_0=\mathbf{0}$. Suppose $\mathbf{X}^{(m)}=(\mathbf{I}-\mathbf{P}_{m-1})\mathbf{X}$ with $\mathbf{z}_1,\ldots,\mathbf{z}_{m-1}$ orthogonal and in $\mathcal{C}$. Then $\mathbf{z}_m=\mathbf{X}^{(m)}\boldsymbol{\phi}_m\in\mathcal{C}\bigl((\mathbf{I}-\mathbf{P}_{m-1})\mathbf{X}\bigr)$, which lies in $\mathcal{C}$ and is orthogonal to $\mathbf{z}_1,\ldots,\mathbf{z}_{m-1}$. If $\mathbf{z}_m\neq\mathbf{0}$, $\mathbf{P}_m=\mathbf{P}_{m-1}+\mathbf{z}_m\mathbf{z}_m'/\|\mathbf{z}_m\|^2$, the right side being symmetric, idempotent (as $\mathbf{z}_m'\mathbf{P}_{m-1}=\mathbf{0}'$) and with column space $\mathcal{C}(\mathbf{z}_1,\ldots,\mathbf{z}_m)$, by the prerequisites' proposition **projections are the symmetric idempotents**; so $\mathbf{X}^{(m+1)}=(\mathbf{I}-\mathbf{z}_m\mathbf{z}_m'/\|\mathbf{z}_m\|^2)(\mathbf{I}-\mathbf{P}_{m-1})\mathbf{X}=(\mathbf{I}-\mathbf{P}_m)\mathbf{X}$. Finally $\mathcal{C}\perp\mathbf{1}$, the columns of $\mathbf{X}$ being centred.

(iii) By (ii) the design $[\mathbf{1}\ \mathbf{z}_1\ \cdots\ \mathbf{z}_M]$ has orthogonal columns, and the normal equations are diagonal, as in Step 1 of the proof of Theorem 5.28.

(iv) Every column of $\mathbf{X}$ is $\mathbf{P}_{m-1}\mathbf{x}_j+(\mathbf{I}-\mathbf{P}_{m-1})\mathbf{x}_j$, so $\mathcal{C}=\mathcal{C}(\mathbf{z}_1,\ldots,\mathbf{z}_{m-1})\oplus\mathcal{C}(\mathbf{X}^{(m)})$, the two parts being orthogonal and both contained in $\mathcal{C}$; hence the projection onto $\mathcal{C}$ is $\mathbf{P}_{m-1}$ plus the projection onto $\mathcal{C}(\mathbf{X}^{(m)})$. If $\mathbf{z}_m=\mathbf{0}$ then $0=\mathbf{Y}'\mathbf{z}_m=\|\mathbf{X}^{(m)\prime}\mathbf{Y}\|^2$, so $\mathbf{Y}\perp\mathcal{C}(\mathbf{X}^{(m)})$ and the projection of $\mathbf{Y}$ onto $\mathcal{C}$ is $\mathbf{P}_{m-1}\mathbf{Y}$. As $\mathbf{1}\perp\mathcal{C}$, the least squares fitted values on $[\mathbf{1}\ \mathbf{X}]$ are $\bar y\mathbf{1}$ plus the projection of $\mathbf{Y}$ onto $\mathcal{C}$, which is PLS($m-1$) by (iii). If $\mathbf{z}_1,\ldots,\mathbf{z}_M$ are nonzero, they are orthogonal, hence independent, and span an $M$-dimensional subspace of $\mathcal{C}$, which is $\mathcal{C}$ when $M=\operatorname{rank}(\mathbf{X})$. ∎

**Remark** (ISLR §6.3.2). PCR chooses its directions without the response and PLS with it: the first PLS direction maximises the covariance with $y$, the first principal component the variance of the predictors. The supervision can reduce bias and adds variance; in practice PLS often does no better than PCR. $M$ is chosen by cross-validation.

**Example 5.6** (PCR and PLS on the Credit data; ISLR §6.3.1, Figure 6.20). The standardised coefficients $\hat{\boldsymbol{\beta}}(M)$ of Theorem 5.28(ii), the share of each component in $R^2$, and ten-fold cross-validation of `pcr` and `plsr`:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
Xs_c <- scale(Xc); sv_c <- svd(Xs_c); th_c <- drop(crossprod(sv_c$u, yc)) / sv_c$d
B_c <- sapply(1:11, function(M) drop(sv_c$v[, 1:M, drop = FALSE] %*% th_c[1:M]))
round(rbind(PVE = sv_c$d^2 / sum(sv_c$d^2), R2_share = (sv_c$d * th_c)^2 / tss_c), 4)
round(sv_c$v[, 10], 3)
set.seed(1); pcr_c <- pcr(Balance ~ ., data = Credit, scale = TRUE, validation = "CV")
set.seed(1); pls_c <- plsr(Balance ~ ., data = Credit, scale = TRUE, validation = "CV")
cvs <- rbind(PCR = MSEP(pcr_c, estimate = "CV")$val[1, 1, ], PLS = MSEP(pls_c, estimate = "CV")$val[1, 1, ])
colnames(cvs) <- 0:11; round(cvs)
c(PCR_min = which.min(cvs[1, ]) - 1, PLS_min = which.min(cvs[2, ]) - 1)
round(drop(R2(pls_c, estimate = "train")$val)[2:4], 4)
signif(c(coef_vs_pcr = max(abs(B_c - drop(coef(pcr_c, ncomp = 1:11)))),
  M11_vs_lm = max(abs(B_c[, 11] / attr(Xs_c, "scaled:scale") - coef(lm(Balance ~ ., data = Credit))[-1]))), 3)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.2, 0.6, 0))
matplot(1:11, t(B_c), type = "l", lty = 1, col = c(2, 4, 3, rep("grey60", 8)), xlab = "number of components",
        ylab = "standardised coefficient")
legend("topleft", c("Income", "Limit", "Rating"), col = c(2, 4, 3), lty = 1, bty = "n", cex = 0.7)
plot(0:11, cvs[1, ] / 1e3, type = "b", pch = 16, cex = 0.6, xlab = "number of components", ylab = "CV MSE (thousands)")
par(op)

*Figure 5.6.* PCR on `Credit`: standardised coefficients against $M$, and ten-fold cross-validation error (ISLR Figure 6.20).

The share of component $m$ in $R^2$ is $(\mathbf{u}_m'\mathbf{Y})^2/\mathrm{TSS}$ (Theorem 5.28(ii)): $0.5807$ for the first, which carries a proportion $0.2505$ of the predictor variance; below $0.06$ for each of the second to the ninth; and $0.2676$ for the tenth, which carries only $0.0238$ of the predictor variance and whose loading vector weights `Income` by $0.836$ against $-0.379$ and $-0.374$ for `Limit` and `Rating`. A direction of small predictor variance is strongly related to `Balance`, the case the remark after Theorem 5.28 describes: the cross-validation error stays above $70{,}000$ up to $M=9$, falls to $10{,}262$ at $M=10$, and is smallest, $10{,}245$, at $M=11$, which is least squares; ISLR's folds put the minimum at $M=10$. PLS reaches $11{,}617$ with three directions and its minimum, $10{,}245$, with ten; its first three directions give $R^2=0.6967$, $0.8653$ and $0.9495$. Theorem 5.28(ii) reproduces the `pcr` coefficients, and $M=11$ the least squares slopes, to rounding error.

## 5 Data Collection

> **Reading.** FREES §6.3. SRM learning outcomes 2(c), 2(e).

### 5.1 Sampling frames and selection

**Definition 5.31** (Sampling frame error and adverse selection; FREES §6.3.1). The **sampling frame** is the list from which the sample is drawn. **Sampling frame error** occurs when the frame does not represent the population of interest; a larger sample from the same frame repeats the error. In insurance, **adverse selection** produces it: applicants who know their risk to be underpriced are more likely to buy, so the insured are not a random sample of the population, as annuitants live longer than the general population.

**Remark** (The **Literary Digest** poll; FREES §6.3.1). In 1936 the **Literary Digest** polled $10$ million voters drawn from telephone books and club lists, received $2.4$ million replies, and predicted Landon over Roosevelt by $57\%$ to $43\%$; Roosevelt won by $62\%$ to $38\%$. In the Depression the frame over-represented upper incomes, which voted differently. Insurers avoid the error by taking the population of interest to be their potential policyholders, pricing annuities from annuitant mortality.

**Proposition 5.32** (Selecting observations by their predictors; FREES §6.3.1, §6.3.5). Let $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ satisfy A1–A3, and let a set $R$ of rows be retained by a rule that uses only the predictors, so that $R$ does not depend on $\boldsymbol{\varepsilon}$, with $\mathbf{X}_R$ of full column rank. Then the retained data satisfy A1–A3 with the same $\boldsymbol{\beta}$ and $\sigma^2$, and A5 if the full data do; their least squares estimator $\hat{\boldsymbol{\beta}}_R$ is unbiased, and $\operatorname{var}(\hat{\boldsymbol{\beta}}_R)-\operatorname{var}(\hat{\boldsymbol{\beta}})$ is positive semi-definite.

**Proof.** $\mathbf{Y}_R=\mathbf{X}_R\boldsymbol{\beta}+\boldsymbol{\varepsilon}_R$, and $\boldsymbol{\varepsilon}_R$, a fixed subvector of $\boldsymbol{\varepsilon}$, has mean $\mathbf{0}$, variance $\sigma^2\mathbf{I}$, and a normal law under A5. Unit 2's theorem **unbiasedness** gives $\operatorname{\mathsf{E}}[\hat{\boldsymbol{\beta}}_R]=\boldsymbol{\beta}$. $\hat{\boldsymbol{\beta}}_R$ is linear in the full $\mathbf{Y}$ (it gives zero weight to the other rows) and unbiased for every $\boldsymbol{\beta}$, so Unit 2's theorem **Gauss–Markov** for the full data gives the variance comparison. ∎

**Lemma 5.33** (Truncated normal moments). Let $Z\sim N(0,1)$ with density $\varphi$ and distribution function $\Phi$, and let $\lambda(a)=\varphi(a)/\{1-\Phi(a)\}$ for $a\in\mathbb{R}$. Then

(i) $\operatorname{\mathsf{E}}[Z\mid Z>a]=\lambda(a)$ and $\lambda(a)>\max(0,a)$;

(ii) $\operatorname{var}(Z\mid Z>a)=1-\lambda(a)\{\lambda(a)-a\}\in(0,1)$;

(iii) $\lambda$ is differentiable with $\lambda'(a)=\lambda(a)\{\lambda(a)-a\}\in(0,1)$;

(iv) $g(a)=\operatorname{\mathsf{E}}[(Z-a)^+]=\varphi(a)-a\{1-\Phi(a)\}$, with $g'(a)=-\{1-\Phi(a)\}$.

**Proof.** **Step 1: two integrals.** $\varphi'(t)=-t\varphi(t)$, so $\int_a^bt\varphi(t)\,\mathrm{d} t=\varphi(a)-\varphi(b)\to\varphi(a)$ as $b\to\infty$, and $\operatorname{\mathsf{E}}[Z\mathbb{1}\{Z>a\}]=\varphi(a)$ by dominated convergence, $|Z|$ being integrable. Integrating by parts, $\int_a^bt^2\varphi(t)\,\mathrm{d} t=a\varphi(a)-b\varphi(b)+\int_a^b\varphi(t)\,\mathrm{d} t$, so $\operatorname{\mathsf{E}}[Z^2\mathbb{1}\{Z>a\}]=a\varphi(a)+1-\Phi(a)$.

**Step 2: (i).** Dividing by $\operatorname{\mathsf{P}}(Z>a)=1-\Phi(a)>0$, $\operatorname{\mathsf{E}}[Z\mid Z>a]=\lambda(a)$, which is positive, and $\lambda(a)-a=\operatorname{\mathsf{E}}[(Z-a)\mathbb{1}\{Z>a\}]/\operatorname{\mathsf{P}}(Z>a)>0$, the integrand being positive on an event of positive probability.

**Step 3: (ii).** By Step 1, $\operatorname{\mathsf{E}}[Z^2\mid Z>a]=1+a\lambda(a)$, so $\operatorname{var}(Z\mid Z>a)=1+a\lambda-\lambda^2=1-\lambda(\lambda-a)$. It is $\int_a^\infty(t-\lambda)^2\varphi(t)\,\mathrm{d} t/\operatorname{\mathsf{P}}(Z>a)>0$, the integrand being positive except at one point; and it is below $1$ because $\lambda>0$ and $\lambda-a>0$.

**Step 4: (iii).** By the quotient rule with $\Phi'=\varphi$, $\lambda'=\{\varphi'(1-\Phi)+\varphi^2\}/(1-\Phi)^2=-a\lambda+\lambda^2$, which lies in $(0,1)$ by Steps 2 and 3.

**Step 5: (iv).** $\operatorname{\mathsf{E}}[(Z-a)^+]=\operatorname{\mathsf{E}}[Z\mathbb{1}\{Z>a\}]-a\operatorname{\mathsf{P}}(Z>a)$, and differentiating, $g'(a)=-a\varphi(a)-\{1-\Phi(a)\}+a\varphi(a)$. ∎

**Proposition 5.34** (Selection by the response flattens the regression; FREES §6.3.3). Let $x$ be a random variable with $\operatorname{\mathsf{E}}[x^2]<\infty$ and $\operatorname{var}(x)>0$, $Z\sim N(0,1)$ independent of $x$, and $y=\beta_0+\beta_1x+\sigma Z$ with $\beta_1>0$, $\sigma>0$. Put $a(u)=(d-\beta_0-\beta_1u)/\sigma$ for a threshold $d$, and $\pi(u)=1-\Phi(a(u))$. For an event $R$ of positive probability write $\operatorname{\mathsf{E}}_R[w]=\operatorname{\mathsf{E}}[w\mathbb{1}_R]/\operatorname{\mathsf{P}}(R)$, $\operatorname{cov}_R(v,w)=\operatorname{\mathsf{E}}_R[vw]-\operatorname{\mathsf{E}}_R[v]\operatorname{\mathsf{E}}_R[w]$ and $\operatorname{var}_R(v)=\operatorname{cov}_R(v,v)$, and call $\operatorname{cov}_R(x,w)/\operatorname{var}_R(x)$ the **population least squares slope** of $w$ on $x$ given $R$, for $w$ with finite second moment.

(i) **Truncation**, retaining only $R=\{y>d\}$: with $m_t(u)=\beta_0+\beta_1u+\sigma\lambda(a(u))$, the mean of the retained responses at $x=u$, $\operatorname{\mathsf{E}}[f(x)\mathbb{1}_R]=\operatorname{\mathsf{E}}[f(x)\pi(x)]$ for every $f$ with $\operatorname{\mathsf{E}}|f(x)|<\infty$ and $\operatorname{\mathsf{E}}[f(x)\,y\mathbb{1}_R]=\operatorname{\mathsf{E}}[f(x)m_t(x)\pi(x)]$ for every $f$ with $\operatorname{\mathsf{E}}[f(x)^2]<\infty$; $0<m_t'<\beta_1$; and the population least squares slope of $y$ on $x$ given $R$ lies in $(0,\beta_1)$.

(ii) **Censoring**, recording $y_c=\max(y,d)$: with $m_c(u)=d+\sigma g(a(u))$, $\operatorname{\mathsf{E}}[f(x)y_c]=\operatorname{\mathsf{E}}[f(x)m_c(x)]$ for every $f$ with $\operatorname{\mathsf{E}}[f(x)^2]<\infty$; $m_c'(u)=\beta_1\{1-\Phi(a(u))\}\in(0,\beta_1)$; and the population least squares slope of $y_c$ on $x$ lies in $(0,\beta_1)$.

(iii) **Selection on $x$**, retaining $R=\{x\in A\}$ with $\operatorname{\mathsf{P}}(R)>0$ and $\operatorname{var}_R(x)>0$: the population least squares slope of $y$ on $x$ given $R$ is $\beta_1$.

**Proof.** **Step 1: integrating out $Z$.** As $x$ and $Z$ are independent, their joint law is the product of their laws, and Fubini–Tonelli gives $\operatorname{\mathsf{E}}[h(x,Z)]=\operatorname{\mathsf{E}}[H(x)]$, with $H(u)=\operatorname{\mathsf{E}}[h(u,Z)]$, whenever $\operatorname{\mathsf{E}}|h(x,Z)|<\infty$. Apply it to $h(u,z)=f(u)\mathbb{1}\{z>a(u)\}$, to $h(u,z)=f(u)(\beta_0+\beta_1u+\sigma z)\mathbb{1}\{z>a(u)\}$ and to $h(u,z)=f(u)\max(\beta_0+\beta_1u+\sigma z,d)=f(u)\{d+\sigma(z-a(u))^+\}$, using $y>d$ iff $Z>a(x)$. The first is bounded by $|f(u)|$; the other two by $|f(u)|$ times $|\beta_0|+|d|+\beta_1|u|+\sigma|z|$, and $\operatorname{\mathsf{E}}|f(x)x|<\infty$ by the Cauchy–Schwarz inequality and $\operatorname{\mathsf{E}}|f(x)Z|=\operatorname{\mathsf{E}}|f(x)|\,\operatorname{\mathsf{E}}|Z|$ by independence. By Lemma 5.33, $\operatorname{\mathsf{P}}(Z>a)=1-\Phi(a)$, $\operatorname{\mathsf{E}}[Z\mathbb{1}\{Z>a\}]=\lambda(a)\{1-\Phi(a)\}$ and $\operatorname{\mathsf{E}}[(Z-a)^+]=g(a)$, so the three functions $H$ are $f(u)\pi(u)$, $f(u)\{\beta_0+\beta_1u+\sigma\lambda(a(u))\}\pi(u)=f(u)m_t(u)\pi(u)$ and $f(u)m_c(u)$. With $a'(u)=-\beta_1/\sigma$, Lemma 5.33(iii) gives $m_t'(u)=\beta_1\{1-\lambda'(a(u))\}\in(0,\beta_1)$, and (iv) gives $m_c'(u)=\beta_1\{1-\Phi(a(u))\}$, in $(0,\beta_1)$ because $0<\Phi<1$.

**Step 2: the slope of a flattened mean.** Let $\operatorname{\mathsf{E}}_Q$ be the expectation under a probability measure $Q$ for which $x$ has finite second moment, positive variance and mean $\mu$, and let $m$ be differentiable with $0<m'<\beta_1$ and $\operatorname{\mathsf{E}}_Q[m(x)^2]<\infty$. By the prerequisites' mean value theorem, $0<(u-\mu)\{m(u)-m(\mu)\}<\beta_1(u-\mu)^2$ for $u\neq\mu$, and, as $\operatorname{\mathsf{E}}_Q[x-\mu]=0$, $\operatorname{cov}_Q(x,m(x))=\operatorname{\mathsf{E}}_Q[(x-\mu)\{m(x)-m(\mu)\}]$. Both $(x-\mu)\{m(x)-m(\mu)\}$ and $\beta_1(x-\mu)^2-(x-\mu)\{m(x)-m(\mu)\}$ are nonnegative and positive on $\{x\neq\mu\}$, which has positive $Q$-probability; hence $0<\operatorname{cov}_Q(x,m(x))<\beta_1\operatorname{var}_Q(x)$.

**Step 3: (i).** Step 1 with $f\equiv1$ gives $\operatorname{\mathsf{P}}(R)=\operatorname{\mathsf{E}}[\pi(x)]>0$, and $\operatorname{\mathsf{E}}_R$ is the expectation under the probability measure $\operatorname{\mathsf{P}}(\,\cdot\cap R)/\operatorname{\mathsf{P}}(R)$. Under it $x$ has positive variance: if $\operatorname{\mathsf{E}}_R[\mathbb{1}\{x\neq c\}]=0$ for a constant $c$, then $\operatorname{\mathsf{E}}[\mathbb{1}\{x\neq c\}\pi(x)]=0$, so $\mathbb{1}\{x\neq c\}\pi(x)=0$ almost surely, and $\pi>0$ gives $x=c$ almost surely, against $\operatorname{var}(x)>0$. The second identity with $f(u)=u$ and $f\equiv1$, then the first with $f(u)=u\,m_t(u)$ and $f=m_t$, give $\operatorname{\mathsf{E}}_R[xy]=\operatorname{\mathsf{E}}_R[x\,m_t(x)]$ and $\operatorname{\mathsf{E}}_R[y]=\operatorname{\mathsf{E}}_R[m_t(x)]$, so $\operatorname{cov}_R(x,y)=\operatorname{cov}_R(x,m_t(x))$. By Lemma 5.33(iii) and the mean value theorem, $0<\lambda(a)\leqslant\lambda(0)+|a|$, so $|m_t(u)|$ is at most a constant plus a multiple of $|u|$; this makes the last two uses of Step 1 legitimate and $\operatorname{\mathsf{E}}_R[m_t(x)^2]<\infty$, and $\operatorname{\mathsf{E}}_R[x^2]\leqslant\operatorname{\mathsf{E}}[x^2]/\operatorname{\mathsf{P}}(R)$. Step 2 applies.

**Step 4: (ii) and (iii).** For censoring, Step 1 with $f(u)=u$ and $f\equiv1$ gives $\operatorname{cov}(x,y_c)=\operatorname{cov}(x,m_c(x))$, and Step 2 applies to $\operatorname{\mathsf{P}}$ itself, since $0\leqslant g(a)\leqslant\operatorname{\mathsf{E}}|Z|+|a|\leqslant1+|a|$ bounds $|m_c(u)|$ by a constant plus a multiple of $|u|$. Under selection on $x$, independence gives $\operatorname{\mathsf{E}}[xZ\mathbb{1}_R]=\operatorname{\mathsf{E}}[x\mathbb{1}\{x\in A\}]\,\operatorname{\mathsf{E}}[Z]=0$ and likewise $\operatorname{\mathsf{E}}[Z\mathbb{1}_R]=0$, so $\operatorname{cov}_R(x,Z)=0$ and $\operatorname{cov}_R(x,y)=\beta_1\operatorname{var}_R(x)$. ∎

**Remark** (Limited dependent variables; FREES §6.3.3). A response is **censored** when values beyond a bound are recorded at the bound, as purchases of zero insurance or claims capped at a policy limit, and **truncated** when such observations are not recorded at all, as losses below a deductible. Truncation removes the observations, censoring keeps their predictors; both bias least squares, and censored and truncated regression models are built for them.

**Example 5.7** (Truncation, censoring and selection on the predictor; FREES §6.3, Figures 6.3–6.4 with the threshold at $6$). $x\sim U(0,10)$, $y=1+x+3Z$ and the threshold $d=6$. The population slopes of Proposition 5.34 are computed by numerical integration, and compared with the mean of $1000$ least squares slopes from samples of $200$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
lam <- function(a) dnorm(a) / pnorm(a, lower.tail = FALSE)
b0 <- 1; b1 <- 1; sg <- 3; d_cut <- 6
a_x <- function(x) (d_cut - b0 - b1 * x) / sg
m_t <- function(x) b0 + b1 * x + sg * lam(a_x(x))
m_c <- function(x) d_cut + sg * (dnorm(a_x(x)) - a_x(x) * pnorm(a_x(x), lower.tail = FALSE))
w_R <- function(x) pnorm(a_x(x), lower.tail = FALSE) / 10
integ <- function(f) integrate(f, 0, 10, rel.tol = 1e-12)$value
p_R <- integ(w_R); E_R <- function(f) integ(function(x) f(x) * w_R(x)) / p_R
slope_t <- (E_R(function(x) x * m_t(x)) - E_R(identity) * E_R(m_t)) /
  (E_R(function(x) x^2) - E_R(identity)^2)
slope_c <- (integ(function(x) x * m_c(x) / 10) - 5 * integ(function(x) m_c(x) / 10)) / (100 / 12)
set.seed(3)
sl <- t(replicate(1000, {
  x <- runif(200, 0, 10); y <- b0 + b1 * x + sg * rnorm(200); R <- y > d_cut; A <- x > 5
  c(all = coef(lm(y ~ x))[[2]], x_above_5 = coef(lm(y[A] ~ x[A]))[[2]],
    truncated = coef(lm(y[R] ~ x[R]))[[2]], censored = coef(lm(pmax(y, d_cut) ~ x))[[2]])
}))
round(rbind(mean_slope = colMeans(sl), mc_se = apply(sl, 2, sd) / sqrt(1000),
            population = c(1, 1, slope_t, slope_c)), 4)
c(P_retained = p_R)

The mean slopes are within one Monte Carlo standard error of the population values: $1$ for all the data and for the selection $x>5$; $0.4625$ under truncation, which retains a proportion $0.5$ of the population; and $0.5000$ under censoring; both of the last two lie in $(0,\beta_1)=(0,1)$, as Proposition 5.34 states.

### 5.2 Sampling regions, omitted variables and missing data

**Proposition 5.35** (Limited sampling regions; FREES §6.3.2). (i) Let $\operatorname{\mathsf{E}}[y_i]=\beta_0+\beta_1x_i+\beta_2x_i^2$ with $\beta_2\neq0$ and at least three distinct $x_i$ and errors satisfying A2–A3, let the line be fitted by least squares, and let $c_0+c_1x$ be the least squares line of $x_i^2$ on $x_i$ over the sample. The fitted line at $x_0$ has bias $-\beta_2\,q(x_0)$, $q(x_0)=x_0^2-c_0-c_1x_0$; $q$ is a quadratic with leading coefficient $1$, $\sum_iq(x_i)=0$, and $|\text{bias}|\to\infty$ as $|x_0|\to\infty$.

(ii) In the line $y_i=\beta_0+\beta_1x_i+\varepsilon_i$, if $s_x=0$ then different $(\beta_0,\beta_1)$ give the same means and $\beta_1$ cannot be estimated; if $s_x>0$, $\operatorname{var}(\hat\beta_1)=\sigma^2/\{(n-1)s_x^2\}$, which grows without bound as $s_x\to0$.

**Proof.** (i) By part (i) of Unit 3's theorem **omitted variables**, with $\mathbf{X}_1=[\mathbf{1}\ \mathbf{x}]$ and the omitted column $(x_i^2)$, the fitted coefficients have expectations $\beta_0+c_0\beta_2$ and $\beta_1+c_1\beta_2$, $(c_0,c_1)'$ being $(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'(x_i^2)$. The fitted line at $x_0$ therefore has expectation $\beta_0+\beta_1x_0+\beta_2(c_0+c_1x_0)$, and subtracting the mean $\beta_0+\beta_1x_0+\beta_2x_0^2$ gives $-\beta_2q(x_0)$. The residuals $q(x_i)$ of a regression with an intercept sum to zero (Unit 2's theorem **residual identities**), and $q(x_0)\geqslant x_0^2-|c_0|-|c_1||x_0|\to\infty$.

(ii) If $s_x=0$, every $x_i$ equals $\bar x$ and $\beta_0+\beta_1x_i=(\beta_0+t\bar x)+(\beta_1-t)x_i$ for every $t$. If $s_x>0$, the variance is Unit 2's corollary **the case $k=1$** on the variances of the least squares coefficients. ∎

**Remark** (Omitted and endogenous variables; FREES §6.3.4). Variables are omitted for lack of data or foresight, or by prohibition, such as race in insurance rating and gender in unisex tables. An omitted variable correlated with the included ones biases their coefficients, by Unit 3's theorem **omitted variables**, and can reverse a sign, as in FREES's refrigerator example. A variable that can be taken as given is **exogenous**; one that cannot, for instance because an omitted variable drives both it and $y$, is **endogenous**. Treating the predictors as random and modelling $\operatorname{\mathsf{E}}[y\mid x_1,\ldots,x_k]$ gives a **sampling-based** model; a **structural** model represents a causal relation and may contain endogenous predictors. FREES's example relates a policyholder's claims $y_{it}$ and premiums $x_{it}$ in two periods:

$$
y_{i2}=\beta_{0,C}+\beta_{1,C}y_{i1}+\beta_{2,C}x_{i2}+\varepsilon_{i1} ,\qquad
x_{i2}=\beta_{0,P}+\beta_{1,P}y_{i1}+\beta_{2,P}x_{i1}+\varepsilon_{i2} ,
$$

so the second-period premium, a predictor in the claims equation, is itself driven by the first period's claims (the error subscripts index the equation, not the period). FREES notes that such **structural equations** models need special estimation techniques, the usual ones being biased. Least squares on the first equation is biased when the two errors are correlated, because $x_{i2}$ then contains $\varepsilon_{i2}$ and so is correlated with the error $\varepsilon_{i1}$ of the equation it enters; with errors independent of each other and of the first-period $y_{i1}$ and $x_{i1}$ it is not.

**Remark** (Redlining, continued; FREES §6.3.4). Harrington and Niehaus found the share of applicants offered substandard contracts positively related to the proportion of Black residents across ZIP codes. Expected claim costs, driven by income, are related to both, and income at the individual level is omitted from the ZIP-code data, so the result does not establish discrimination in access.

**Definition 5.36** (Missing data; FREES §6.3.5). Data are **missing at random**, in FREES's sense, when the reason for their absence is unrelated to their values. Common strategies are to delete the incomplete observations, to drop a variable that is mostly missing, and to **impute**, filling in values predicted from the observed data.

**Proposition 5.37** (Mean imputation understates variability; FREES §6.3.5). Let $n_o\geqslant2$ of $n$ values of a variable be observed, with mean $\bar x_o$ and sample variance $s_o^2$, and let the $n-n_o$ missing values be replaced by $\bar x_o$. The completed data have mean $\bar x_o$ and sample variance $s_o^2(n_o-1)/(n-1)$, smaller than $s_o^2$ when $n>n_o$.

**Proof.** The completed sum is $n_o\bar x_o+(n-n_o)\bar x_o=n\bar x_o$. The imputed values contribute zero to $\sum_i(x_i-\bar x_o)^2$, which is therefore $(n_o-1)s_o^2$, and dividing by $n-1$ gives the variance. ∎

**Remark** (FREES §6.3.5). Deleting incomplete observations is Proposition 5.32 when the missingness depends on the predictors only, and costs precision; when it depends on the response, the deletion is a truncation, of which Proposition 5.34 treats the simplest case. Segal's study of life insurers' expenses, which FREES cites, kept $448$ of $733$ firm-years by rules on the explanatory variables, a selection of the first kind.

**Example 5.8** (Extrapolation, truncation and imputation; FREES Figures 6.1, 6.4). A quadratic mean $2+3x-0.5x^2$ sampled on $1\leqslant x\leqslant3$ and a line fitted; the truncated sample of Example 5.7; and a mean imputation:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.5)
set.seed(5); xr <- runif(50, 1, 3); yr <- 2 + 3 * xr - 0.5 * xr^2 + rnorm(50, sd = 0.25)
cc <- coef(lm(I(xr^2) ~ xr)); q_x <- function(x0) x0^2 - cc[[1]] - cc[[2]] * x0
round(c(bias_at_2 = 0.5 * q_x(2), bias_at_8 = 0.5 * q_x(8), sum_q = sum(q_x(xr))), 4)
set.seed(4); xm <- rnorm(50); xm[sample(50, 15)] <- NA
x_imp <- ifelse(is.na(xm), mean(xm, na.rm = TRUE), xm)
c(var_observed = var(xm, na.rm = TRUE), var_imputed = var(x_imp),
  ratio = var(x_imp) / var(xm, na.rm = TRUE), formula = (35 - 1) / (50 - 1))
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 0.6, 0.6), mgp = c(2, 0.6, 0))
plot(xr, yr, xlim = c(0, 8), ylim = c(-12, 8), pch = 16, cex = 0.4, xlab = "x", ylab = "y")
curve(2 + 3 * x - 0.5 * x^2, add = TRUE, lty = 2); abline(lm(yr ~ xr), col = "red")
abline(v = c(1, 3), col = "grey70", lty = 3)
set.seed(6); xt <- runif(300, 0, 10); yt <- 1 + xt + 3 * rnorm(300); kt <- yt > 6
plot(xt, yt, pch = 16, cex = 0.4, col = ifelse(kt, "black", "grey70"), xlab = "x", ylab = "y")
abline(1, 1, lty = 2); abline(lm(yt[kt] ~ xt[kt]), col = "red"); abline(lm(pmax(yt, 6) ~ xt), col = "blue")
abline(h = 6, col = "grey70", lty = 3)
par(op)

*Figure 5.7.* Left: a line fitted on $1\leqslant x\leqslant3$ (red) to a quadratic mean (dashed), extrapolated (FREES Figure 6.1). Right: the true line (dashed), the least squares line of the observations above $d=6$ (red; grey points are lost), and of the censored responses $\max(y,6)$ (blue) (FREES Figure 6.4).

The bias $-\beta_2q(x_0)$ of Proposition 5.35(i) is $-0.1651$ at $x_0=2$, inside the sampled range, and $17.7999$ at $x_0=8$, and the $q(x_i)$ sum to zero. The imputed variable has variance $0.5543$ against $0.7989$ for the $35$ observed values, the ratio $0.6939=34/49$ of Proposition 5.37.

## 6 Computation in R

> **Reading.** ISLR §6.5.1, §6.5.3, §12.5.1 · FREES chapter 5 script.

### 6.1 The ISLR lab: subset selection

`Salary` is missing for $59$ of the $322$ players in `Hitters`; the rows with missing values are removed, and `regsubsets` finds the best model of each size, up to eight predictors by default:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
dim(Hitters); sum(is.na(Hitters$Salary))
Hitters <- na.omit(Hitters); dim(Hitters); sum(is.na(Hitters))
regfit.full <- regsubsets(Salary ~ ., Hitters)
w8 <- summary(regfit.full)$which[, -1]
for (d in 1:8) cat(d, ":", names(which(w8[d, ])), "\n")
regfit.full <- regsubsets(Salary ~ ., data = Hitters, nvmax = 19)
reg.summary <- summary(regfit.full)
names(reg.summary)
round(reg.summary$rsq, 4)
c(adjr2_max = which.max(reg.summary$adjr2), cp_min = which.min(reg.summary$cp),
  bic_min = which.min(reg.summary$bic))
round(reg.summary$bic, 1)
s2_full <- summary(lm(Salary ~ ., data = Hitters))$sigma^2
c(ISLR_Cp_min = which.min(reg.summary$rss + 2 * (1:19) * s2_full),
  ISLR_BIC_min = which.min(reg.summary$rss + log(nrow(Hitters)) * (1:19) * s2_full))
coef(regfit.full, 6)

The best two-predictor model contains `Hits` and `CRBI`. $R^2$ rises from $0.3215$ with one predictor to $0.5461$ with all nineteen. $R_a^2$ selects $11$ predictors, $C_p^{\mathrm F}$ $10$ and the `regsubsets` BIC $6$; ISLR's linear $C_p$ and BIC also select $10$ and $6$, BIC no more than $C_p$ as Proposition 5.14(ii) requires of these two forms. The BIC model contains `AtBat`, `Hits`, `Walks`, `CRBI`, `DivisionW` and `PutOuts`.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.4)
op <- par(mfrow = c(2, 2), mar = c(3.0, 3.8, 0.6, 0.6), mgp = c(1.9, 0.6, 0))
plot(reg.summary$rss / 1e6, xlab = "", ylab = "RSS (millions)", type = "l")
plot(reg.summary$adjr2, xlab = "", ylab = "Adjusted RSq", type = "l")
points(11, reg.summary$adjr2[11], col = "red", cex = 2, pch = 20)
plot(reg.summary$cp, xlab = "Number of Variables", ylab = "Cp", type = "l")
points(10, reg.summary$cp[10], col = "red", cex = 2, pch = 20)
plot(reg.summary$bic, xlab = "Number of Variables", ylab = "BIC", type = "l")
points(6, reg.summary$bic[6], col = "red", cex = 2, pch = 20)
par(op)

*Figure 5.8.* $\mathrm{RSS}$, $R_a^2$, $C_p^{\mathrm F}$ and the `regsubsets` BIC of the best `Hitters` model of each size, with the selected sizes in red.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.6)
op <- par(cex = 0.8)
plot(regfit.full, scale = "bic", mar = c(6.2, 3.6, 0.6, 0.6))
par(op)

*Figure 5.9.* `plot(regfit.full, scale = "bic")`: each row is the best model of one size, its predictors shaded, the rows ordered by BIC.

Seven models have a BIC between $-141.8$ and $-147.9$; the top row is the six-predictor model, at $-147.9$. Forward and backward stepwise selection:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
regfit.fwd <- regsubsets(Salary ~ ., data = Hitters, nvmax = 19, method = "forward")
regfit.bwd <- regsubsets(Salary ~ ., data = Hitters, nvmax = 19, method = "backward")
same_as_best <- function(fit) sapply(1:19, function(d)
  identical(summary(fit)$which[d, ], reg.summary$which[d, ]))
agree <- rbind(forward = same_as_best(regfit.fwd), backward = same_as_best(regfit.bwd))
colnames(agree) <- 1:19; agree[, 1:9]
coef(regfit.full, 7); coef(regfit.fwd, 7); coef(regfit.bwd, 7)

Forward selection agrees with best subset selection for one to six predictors, as the lab states; at seven the three procedures give three different models. The validation set approach, with the search run on the training observations only, and the lab's own `predict` method:

In [ ]:
set.seed(1)
train <- sample(c(TRUE, FALSE), nrow(Hitters), replace = TRUE)
test <- (!train)
regfit.best <- regsubsets(Salary ~ ., data = Hitters[train, ], nvmax = 19)
test.mat <- model.matrix(Salary ~ ., data = Hitters[test, ])
val.errors <- rep(NA, 19)
for (i in 1:19) {
  coefi <- coef(regfit.best, id = i)
  pred <- test.mat[, names(coefi)] %*% coefi
  val.errors[i] <- mean((Hitters$Salary[test] - pred)^2)
}
round(val.errors)
which.min(val.errors)
coef(regfit.best, 7)
predict.regsubsets <- function(object, newdata, id, ...) {
  form <- as.formula(object$call[[2]])
  mat <- model.matrix(form, newdata)
  coefi <- coef(object, id = id)
  xvars <- names(coefi)
  mat[, xvars] %*% coefi
}
regfit.best <- regsubsets(Salary ~ ., data = Hitters, nvmax = 19)
coef(regfit.best, 7)

The validation error is smallest at seven predictors; the best seven-predictor model on all the data differs from the one found on the training half. Ten-fold cross-validation, with best subset selection inside each training set:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
k <- 10
n <- nrow(Hitters)
set.seed(1)
folds <- sample(rep(1:k, length = n))
cv.errors <- matrix(NA, k, 19, dimnames = list(NULL, paste(1:19)))
for (j in 1:k) {
  best.fit <- regsubsets(Salary ~ ., data = Hitters[folds != j, ], nvmax = 19)
  for (i in 1:19) {
    pred <- predict(best.fit, Hitters[folds == j, ], id = i)
    cv.errors[j, i] <- mean((Hitters$Salary[folds == j] - pred)^2)
  }
}
mean.cv.errors <- apply(cv.errors, 2, mean)
round(mean.cv.errors)
which.min(mean.cv.errors)
op <- par(mar = c(3.2, 3.8, 0.6, 0.6), mgp = c(2.2, 0.6, 0))
plot(mean.cv.errors, type = "b", xlab = "number of predictors", ylab = "CV error")
par(op)
reg.best <- regsubsets(Salary ~ ., data = Hitters, nvmax = 19)
coef(reg.best, 10)

*Figure 5.10.* Ten-fold cross-validation error of best subset selection on `Hitters`.

Cross-validation selects ten predictors, as in the lab.

### 6.2 The ISLR lab: PCR and PLS

The matrix `x`, the response `y` and the half-sample `train` of ISLR §6.5.2:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
x <- model.matrix(Salary ~ ., Hitters)[, -1]
y <- Hitters$Salary
set.seed(1)
train <- sample(1:nrow(x), nrow(x) / 2)
test <- (-train)
y.test <- y[test]
set.seed(2)
pcr.fit <- pcr(Salary ~ ., data = Hitters, scale = TRUE, validation = "CV")
rmsep <- RMSEP(pcr.fit, estimate = "CV")$val[1, 1, ]
round(rmsep[c(1:5, 19, 20)], 1); c(MSE_of_352.8 = 352.8^2, cv_min_M = which.min(rmsep) - 1)
round(explvar(pcr.fit)[1:5], 2)
set.seed(1)
pcr.fit <- pcr(Salary ~ ., data = Hitters, subset = train, scale = TRUE, validation = "CV")
which.min(RMSEP(pcr.fit, estimate = "CV")$val[1, 1, ]) - 1
pcr.pred <- predict(pcr.fit, x[test, ], ncomp = 5)
mean((pcr.pred - y.test)^2)
pcr.fit <- pcr(y ~ x, scale = TRUE, ncomp = 5)
round(drop(R2(pcr.fit, estimate = "train")$val)[-1], 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
op <- par(mar = c(3.2, 3.8, 1.6, 0.6), mgp = c(2.2, 0.6, 0))
set.seed(2); validationplot(pcr(Salary ~ ., data = Hitters, scale = TRUE, validation = "CV"),
                            val.type = "MSEP", main = "")
par(op)

*Figure 5.11.* Ten-fold cross-validation mean squared error of PCR on `Hitters` against the number of components: the cross-validation estimate (solid) and its bias-corrected version `adjCV` (dashed).

The root mean squared cross-validation error is $452.0$ with no component; the lab's example $352.8$, the value at four components, is the MSE $124{,}468$. The minimum, $339.2$, is at $M=18$, one short of least squares ($339.6$ at $M=19$, Theorem 5.28(v)), and one component gives $351.9$. One component explains $38.31\%$ of the variance of the predictors. On the training half, cross-validation selects $M=5$, with test MSE $142{,}812$; refitted on all the data, the five components explain proportions $0.4063$ to $0.4490$ of the variance of `Salary` as they are added, each adding its share (Theorem 5.28(ii)).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
set.seed(1)
pls.fit <- plsr(Salary ~ ., data = Hitters, subset = train, scale = TRUE, validation = "CV")
which.min(RMSEP(pls.fit, estimate = "CV")$val[1, 1, ]) - 1
pls.pred <- predict(pls.fit, x[test, ], ncomp = 1)
mean((pls.pred - y.test)^2)
pls.fit <- plsr(Salary ~ ., data = Hitters, scale = TRUE, ncomp = 1)
round(c(X_percent = explvar(pls.fit),
  Salary_percent = 100 * drop(R2(pls.fit, estimate = "train")$val)[2]), 2)

Cross-validation on the training half selects one PLS direction, with test MSE $151{,}995$, above PCR's $142{,}812$. On all the data the direction explains $38.08\%$ of the variance of the predictors, less than the first principal component's $38.31\%$, and $43.05\%$ of the variance of `Salary`, more than the first principal component's $40.63\%$ and close to the five-component PCR's $44.90\%$: the first PLS direction maximises the covariance with the response, the first principal component the variance of the predictors (Proposition 5.30(i)).

### 6.3 The ISLR lab: principal components

In [ ]:
states <- row.names(USArrests)
names(USArrests)
apply(USArrests, 2, mean)
apply(USArrests, 2, var)
pr.out <- prcomp(USArrests, scale = TRUE)
names(pr.out)
pr.out$center
pr.out$scale
pr.out$rotation
dim(pr.out$x)
pr.out$sdev
pr.var <- pr.out$sdev^2
pve <- pr.var / sum(pr.var)
pve
a <- c(1, 2, 8, -3)
cumsum(a)

`prcomp` centres by default and standardises with `scale = TRUE`; its `rotation` is the matrix of loadings and `x` the $50\times4$ matrix of scores. The means range from $7.788$ to $170.760$ and the variances from $18.97$ to $6945.17$. Here `prcomp` returns the first loading vector with negative entries and the second with a positive `UrbanPop` loading, so the lab's flip of both signs would reverse the second relative to ISLR Table 12.1; by Theorem 5.20(v) only the signs are arbitrary. The proportions of variance explained are $0.62006$, $0.24744$, $0.08914$ and $0.04336$. The biplot, with the signs of ISLR Figure 12.1, is the left panel of the figure in Example 5.5.

### 6.4 FREES's chapter 5 script

The script fits the data of Example 5.2 with $1$, $3$, $10$ and $50$ candidates and compares the nested fits:

In [ ]:
m1 <- lm(y ~ obs1, data = snoop)
m3 <- lm(y ~ obs1 + obs2 + obs3, data = snoop)
m10 <- lm(reformulate(paste0("obs", 1:10), "y"), data = snoop)
round(cor(snoop[, c(paste0("obs", 1:10), "y")])["y", 1:10], 3)
sapply(list(m1 = m1, m3 = m3, m10 = m10, m50 = full_s), function(m) round(fit_summary(m)[c("F", "p_value")], 4))
round(sapply(list(a = anova(m1, m3), b = anova(m3, m10), c = anova(m10, full_s)), function(t) t$`Pr(>F)`[2]), 4)
round(coef(summary(lm(y ~ obs27 + obs29 + obs32, data = snoop))), 4)

No correlation of $y$ with the first ten candidates exceeds $0.167$ in absolute value; none of the fits with $1$, $3$, $10$ and $50$ candidates has a significant $F$ test ($p$ from $0.1991$ to $0.3482$), and neither has any of the nested comparisons ($p$ from $0.1702$ to $0.4786$). The three variables kept in Example 5.2 have $t$ ratios $2.1662$, $2.4435$ and $2.5848$.

### 6.5 Formulas against package output

For `Hitters`: the `regsubsets` statistics from their formulas; ISLR's $C_p$ and BIC; R's `AIC`, `BIC` and `extractAIC` and FREES's $\mathrm{AIC}_{\mathrm F}$ (Proposition 5.13 and the remark on software forms); the likelihood ratio of Proposition 5.15; the forward step of Proposition 5.4; and *PRESS* (Proposition 5.17):

In [ ]:
nH <- nrow(Hitters); dH <- 1:19; tssH <- sum((Hitters$Salary - mean(Hitters$Salary))^2)
s2H <- summary(lm(Salary ~ ., data = Hitters))$sigma^2; rssH <- reg.summary$rss
signif(c(cp = max(abs(reg.summary$cp - (rssH / s2H - nH + 2 * (dH + 1)))),
  bic = max(abs(reg.summary$bic - (nH * log(rssH / tssH) + (dH + 1) * log(nH)))),
  adjr2 = max(abs(reg.summary$adjr2 - (1 - (rssH / (nH - dH - 1)) / (tssH / (nH - 1))))),
  rsq = max(abs(reg.summary$rsq - (1 - rssH / tssH)))), 3)
c(ISLR_Cp_min = which.min((rssH + 2 * dH * s2H) / nH), ISLR_BIC_min = which.min((rssH + log(nH) * dH * s2H) / nH))
m6 <- lm(Salary ~ AtBat + Hits + Walks + CRBI + Division + PutOuts, data = Hitters)
rss6 <- deviance(m6); ll6 <- -nH / 2 * (log(2 * pi * rss6 / nH) + 1)
signif(c(logLik = as.numeric(logLik(m6)) - ll6, AIC = AIC(m6) - (-2 * ll6 + 2 * (6 + 2)),
  BIC = BIC(m6) - (-2 * ll6 + log(nH) * (6 + 2)),
  extractAIC = extractAIC(m6)[2] - (nH * log(rss6 / nH) + 2 * 7),
  AIC_F_minus_AIC = (nH * log(rss6 / (nH - 7)) + nH * log(2 * pi) + nH + 3 + 6) - AIC(m6) -
    (nH * log(nH / (nH - 7)) - 7)), 3)
m10 <- lm(Salary ~ AtBat + Hits + Walks + CAtBat + CRuns + CRBI + CWalks + Division + PutOuts + Assists,
          data = Hitters)
F610 <- anova(m6, m10)$F[2]; LR <- nH * log(rss6 / deviance(m10))
round(c(F = F610, LR = LR), 3)
c(LR_from_F = signif(LR - nH * log(1 + 4 * F610 / (nH - 10 - 1)), 3),
  AIC_prefers_10 = AIC(m10) < AIC(m6), LR_above_8 = LR > 8)
cur <- c("Hits", "CRBI"); cand <- setdiff(names(Hitters), c("Salary", cur))
st <- sapply(cand, function(v) { m <- lm(reformulate(c(cur, v), "Salary"), data = Hitters)
  c(rss = deviance(m), t2 = tail(coef(summary(m))[, 3], 1)[[1]]^2) })
rss2 <- deviance(lm(Salary ~ Hits + CRBI, data = Hitters))
c(argmin_RSS = colnames(st)[which.min(st["rss", ])], argmax_t2 = colnames(st)[which.max(st["t2", ])])
signif(max(abs(st["t2", ] - (nH - 2 - 2) * (rss2 / st["rss", ] - 1))), 3)
press_formula <- sum((residuals(m6) / (1 - hatvalues(m6)))^2)
press_refit <- sum(sapply(1:nH, function(i) (Hitters$Salary[i] -
  predict(update(m6, data = Hitters[-i, ]), Hitters[i, ]))^2))
signif(c(PRESS = press_formula, formula_vs_refit = press_formula - press_refit), 6)

The `regsubsets` statistics equal their formulas, and ISLR's $C_p$ and BIC select $10$ and $6$ predictors, as the `regsubsets` forms do. The likelihood identities of Proposition 5.13 and the remark on software forms hold to rounding error. For the six- and ten-predictor models, $F=4.357$ and $\mathrm{LR}=17.588=n\ln\{1+4F/(n-11)\}$, and `AIC` prefers the larger model, as $\mathrm{LR}>2q=8$ requires (Proposition 5.15). From the model with `Hits` and `CRBI`, the candidate with the smallest $\mathrm{RSS}$ and the one with the largest $t^2$ are both `PutOuts`, and $t^2$ follows the formula of Proposition 5.4(i). *PRESS* from the leverages equals the value from $263$ refits. For `USArrests` and `Hitters`: principal components by `eigen`, `svd` and `prcomp` (Theorem 5.20, Proposition 5.23), the best approximation of Theorem 5.22, PCR by Theorem 5.28 against `pcr`, and PLS by the deflation of Definition 5.29 against `plsr`:

In [ ]:
Xu <- scale(USArrests); nu <- nrow(Xu); Su <- crossprod(Xu) / nu
eu <- eigen(Su); su <- svd(Xu); pu <- prcomp(USArrests, scale = TRUE)
signif(c(eigen_vs_prcomp = max(abs(abs(eu$vectors) - abs(pu$rotation))),
  svd_vs_prcomp = max(abs(abs(su$v) - abs(pu$rotation))),
  lambda_vs_d2_over_n = max(abs(eu$values - su$d^2 / nu)),
  lambda_vs_sdev2 = max(abs(eu$values - pu$sdev^2 * (nu - 1) / nu)),
  scores_vs_UD = max(abs(abs(pu$x) - abs(su$u %*% diag(su$d)))),
  rank2_error = sum((Xu - pu$x[, 1:2] %*% t(pu$rotation[, 1:2]))^2) - nu * sum(eu$values[3:4])), 3)
set.seed(7); Q <- qr.Q(qr(matrix(rnorm(8), 4)))
round(c(trace_random_Q = sum(diag(t(Q) %*% Su %*% Q)), lambda1_plus_lambda2 = sum(eu$values[1:2])), 4)
XH <- scale(x); svH <- svd(XH); thH <- drop(crossprod(svH$u, y)) / svH$d
pcrH <- pcr(Salary ~ ., data = Hitters, scale = TRUE, ncomp = 19)
fit_pcr <- function(M) mean(y) + drop(svH$u[, 1:M, drop = FALSE] %*% crossprod(svH$u[, 1:M, drop = FALSE], y))
b_pcr <- function(M) drop(svH$v[, 1:M, drop = FALSE] %*% thH[1:M])
signif(c(fitted_M5 = max(abs(fit_pcr(5) - drop(fitted(pcrH)[, 1, 5]))),
  coef_M5 = max(abs(b_pcr(5) - drop(coef(pcrH, ncomp = 5)))),
  M19_vs_lm = max(abs(fit_pcr(19) - fitted(lm(Salary ~ ., data = Hitters))))), 3)
pls_manual <- function(X, Y, M) { Z <- matrix(0, nrow(X), M)
  for (m in 1:M) { z <- X %*% crossprod(X, Y); Z[, m] <- z; X <- X - z %*% crossprod(z, X) / sum(z^2) }
  Z }
ZH <- pls_manual(XH, y, 19); plsH <- plsr(Salary ~ ., data = Hitters, scale = TRUE, ncomp = 19)
fit_pls <- function(M) mean(y) + drop(ZH[, 1:M, drop = FALSE] %*% (crossprod(ZH[, 1:M, drop = FALSE], y) / colSums(ZH[, 1:M, drop = FALSE]^2)))
wH <- drop(crossprod(XH, y))
signif(c(weight1 = max(abs(abs(plsH$loading.weights[, 1]) - abs(wH) / sqrt(sum(wH^2)))),
  fitted_M1 = max(abs(fit_pls(1) - fitted(plsH)[, 1, 1])),
  fitted_M3 = max(abs(fit_pls(3) - fitted(plsH)[, 1, 3])),
  scores_orthogonal = max(abs(crossprod(ZH[, 1:5])[upper.tri(diag(5))])) / max(colSums(ZH[, 1:5]^2)),
  M19_vs_lm = max(abs(fit_pls(19) - fitted(lm(Salary ~ ., data = Hitters))))), 3)

`eigen`, `svd` and `prcomp` agree up to sign and the divisor, and the error of the rank-two approximation is $n(\lambda_3+\lambda_4)$, to rounding error; for a random $4\times2$ matrix with orthonormal columns $\operatorname{tr}(\mathbf{Q}'\mathbf{S}\mathbf{Q})=1.780$, below $\lambda_1+\lambda_2=3.401$ (Theorem 5.22(i)). The formulas of Theorem 5.28 reproduce the fitted values and coefficients of `pcr`, and with $19$ components least squares. The deflation of Definition 5.29 reproduces the first weight vector and the fitted values of `plsr`, its scores are orthogonal, and with $19$ directions it gives least squares (Proposition 5.30(iv)).

## 7 Exercises

**Exercise 5.1** (FREES exercise 5.3, part (b)). `UNLifeExpectancy`. Use a stepwise regression algorithm to select a model for `LIFEEXP`, leaving out `RESEARCHERS`, `SMOKING` and `FEMALEBOSS`, which have many missing values, and using only the countries with no missing values; do it with and without the categorical variable `REGION`.

**Solution.** The candidates are the remaining variables, with `GDP` and `PRIVATEHEALTH` replaced by their logarithms as in part (a), set in Unit 4; `REGION` enters as one factor, all its indicators together. `step` runs the stepwise algorithm from the full model with the AIC form $n\ln(\mathrm{RSS}/n)+2(d+1)$, and with $k=\ln n$ for the BIC form:

In [ ]:
un <- read.csv("data/frees/csv/UNLifeExpectancy.csv")
un$lnGDP <- log(un$GDP); un$lnHEALTH <- log(un$PRIVATEHEALTH)
v_un <- c("LIFEEXP", "ILLITERATE", "POP", "FERTILITY", "lnHEALTH", "PUBLICEDUCATION",
          "HEALTHEXPEND", "BIRTHATTEND", "PHYSICIAN", "lnGDP", "REGION")
d_un <- na.omit(un[v_un]); d_un$REGION <- factor(d_un$REGION); nrow(d_un)
no_region <- lm(LIFEEXP ~ . - REGION, data = d_un); with_region <- lm(LIFEEXP ~ ., data = d_un)
formula(step(no_region, trace = 0)); formula(step(with_region, trace = 0))
formula(step(no_region, trace = 0, k = log(nrow(d_un))))
formula(step(with_region, trace = 0, k = log(nrow(d_un))))
formula(step(lm(LIFEEXP ~ 1, data = d_un), scope = formula(with_region), trace = 0))

$131$ countries have complete records. Without `REGION`, the AIC and BIC forms both select `FERTILITY`, `PUBLICEDUCATION`, `HEALTHEXPEND` and `BIRTHATTEND`. With `REGION`, the AIC form adds `REGION` and `ILLITERATE` to these four, and the BIC form, which penalises size more (Proposition 5.14(ii)), keeps `REGION`, `ILLITERATE`, `FERTILITY`, `PUBLICEDUCATION` and `BIRTHATTEND`. Forward selection from the null model reaches the same six terms as the search from the full model. `lnGDP`, `lnHEALTH`, `POP` and `PHYSICIAN` are dropped in every run; each selected model is a candidate for the diagnostic checking of Unit 4, not a final answer. ■

**Exercise 5.2** (FREES exercise 5.4, part (d)). The $275$ term life purchasers, with candidates `LNINCOME`, `EDUCATION`, `NUMHH`, `MARSTAT`, `AGE` and `GENDER` for `LNFACE`. A stepwise regression algorithm suggests `LNINCOME`, `EDUCATION`, `NUMHH` and `GENDER`. (i) What is the purpose of stepwise regression? (ii) Describe two important drawbacks of stepwise regression algorithms.

**Solution.**

In [ ]:
term <- subset(read.csv("data/frees/csv/TermLife.csv"), FACE > 0)
term$LNFACE <- log(term$FACE); term$LNINCOME <- log(term$INCOME); term$MARSTAT <- factor(term$MARSTAT)
full_t <- lm(LNFACE ~ LNINCOME + EDUCATION + NUMHH + MARSTAT + AGE + GENDER, data = term)
nobs(full_t); formula(step(full_t, trace = 0)); formula(step(full_t, trace = 0, k = log(nobs(full_t))))
term$MAR1 <- as.numeric(term$MARSTAT == 1); term$MAR2 <- as.numeric(term$MARSTAT == 2)
formula(step(lm(LNFACE ~ LNINCOME + EDUCATION + NUMHH + MAR1 + MAR2 + AGE + GENDER, data = term), trace = 0))

With `MARSTAT` as one factor, `step` selects the four variables of the statement under both the AIC and the BIC form; with its two indicators as separate candidates, the AIC form also keeps `MAR2`. (i) Stepwise regression searches quickly through the candidate models and suggests one or a few for closer study, adding or deleting one variable at a time; by Proposition 5.4 each step is the one that most reduces, or least increases, the residual sum of squares. (ii) First, it snoops: the chosen variables are those whose $t$ ratios happened to be large, so the $t$ and $F$ ratios of the final model overstate their significance, and a cutoff has no meaning as a significance level (Proposition 5.6, Example 5.2). Second, the result need not be the best model: it does not search all $2^6$ subsets, judges variables one at a time and may miss jointly important ones, and ignores nonlinear terms, outliers, high leverage points and the analyst's knowledge. ■

**Exercise 5.3** (ISLR §6.6, exercise 1). Best subset, forward stepwise and backward stepwise selection each give models with $0,1,\ldots,p$ predictors. (a) Which of the three models with $k$ predictors has the smallest training $\mathrm{RSS}$? (b) The smallest test $\mathrm{RSS}$? (c) True or false: the predictors of the $k$-predictor model identified by (i) forward selection are a subset of those of the $(k+1)$-predictor model identified by forward selection; (ii) backward, of backward; (iii) backward, of forward; (iv) forward, of backward; (v) best subset, of best subset.

**Solution.** (a) Best subset selection, by Proposition 5.5(iv): its $\mathcal{M}_k$ minimises $\mathrm{RSS}$ over all subsets of size $k$, including the other two; they may tie. (b) No procedure is guaranteed to win: test $\mathrm{RSS}$ is not what any of them minimises, and the best subset model, chosen from the most candidates, is the most exposed to overfitting. (c) (i) and (ii) are true, by Proposition 5.5(ii). (iii), (iv) and (v) are false in general; on the `Credit` data:

In [ ]:
w_c <- summary(regsubsets(Balance ~ ., data = Credit, nvmax = 11, method = "backward"))
rbind(backward_1 = vars(w_c$which, 1), forward_2 = vars(f_c$which, 2),
      forward_1 = vars(f_c$which, 1), backward_2 = vars(w_c$which, 2),
      best_3 = vars(b_c$which, 3), best_4 = vars(b_c$which, 4))

The backward one-predictor model $\{\texttt{Limit}\}$ is not in the forward two-predictor model, the forward one-predictor model $\{\texttt{Rating}\}$ is not in the backward two-predictor model, and the best three-predictor model contains `Rating`, which the best four-predictor model does not. ■

**Exercise 5.4** (ISLR §6.6, exercise 8, parts (a)–(d), and (f) without the lasso). (a) Generate a predictor $X$ and a noise vector $\varepsilon$ of length $n=100$ with `rnorm`. (b) Generate $Y=\beta_0+\beta_1X+\beta_2X^2+\beta_3X^3+\varepsilon$. (c) Use best subset selection among $X,X^2,\ldots,X^{10}$: which model do $C_p$, BIC and adjusted $R^2$ choose? Show plots and report its coefficients. (d) Repeat with forward and backward stepwise selection. (f) Generate $Y=\beta_0+\beta_7X^7+\varepsilon$ and perform best subset selection.

**Solution.** With $\beta_0=3$, $\beta_1=2$, $\beta_2=-3$, $\beta_3=0.3$ and $\beta_7=7$. The sizes chosen by ISLR's $C_p$ and BIC, with $\hat\sigma^2$ from the model with all ten powers; by the `regsubsets` statistics $C_p^{\mathrm F}$, which ranks all models as $C_p$ does (Proposition 5.13(i)), and the log form of BIC, which ranks models of equal size as BIC does (Proposition 5.14(i)); and by $R_a^2$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
set.seed(1); X8 <- rnorm(100); eps8 <- rnorm(100)
Y8 <- 3 + 2 * X8 - 3 * X8^2 + 0.3 * X8^3 + eps8
D8 <- data.frame(y = Y8, poly(X8, 10, raw = TRUE)); names(D8) <- c("y", paste0("x", 1:10))
choose <- function(f, s2) { s <- summary(f); d <- seq_along(s$rss)
  c(Cp = which.min(s$rss + 2 * d * s2), BIC = which.min(s$rss + log(100) * d * s2),
    Cp_F = which.min(s$cp), BIC_log = which.min(s$bic), adjR2 = which.max(s$adjr2)) }
s2_8 <- summary(lm(y ~ ., data = D8))$sigma^2
fits8 <- lapply(c("exhaustive", "forward", "backward"), function(m)
  regsubsets(y ~ ., data = D8, nvmax = 10, method = m))
t(sapply(fits8, choose, s2 = s2_8))
lapply(fits8, function(f) round(coef(f, 3), 4))
s8 <- summary(fits8[[1]]); d8 <- 1:10
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.8, 0.6, 0.4), mgp = c(2.2, 0.6, 0))
plot((s8$rss + 2 * d8 * s2_8) / 100, type = "b", pch = 16, cex = 0.6, xlab = "number of predictors", ylab = "Cp")
plot((s8$rss + log(100) * d8 * s2_8) / 100, type = "b", pch = 16, cex = 0.6, xlab = "number of predictors", ylab = "BIC")
plot(s8$adjr2, type = "b", pch = 16, cex = 0.6, xlab = "number of predictors", ylab = "adjusted R-squared")
par(op)
D8$y <- 3 + 7 * X8^7 + eps8
s2_7 <- summary(lm(y ~ ., data = D8))$sigma^2
f7 <- regsubsets(y ~ ., data = D8, nvmax = 10); choose(f7, s2_7)
round(coef(f7, 1), 4); round(coef(f7, 2), 4)

*Figure 5.12.* ISLR's $C_p$ and BIC and $R_a^2$ of the best model of each size for the cubic of part (b).

All five criteria choose three predictors under each search. Best subset and forward selection choose $X$, $X^2$ and $X^7$, with coefficients $2.3562$, $-3.1651$ and $0.0105$, and backward selection $X$, $X^2$ and $X^9$: the small cubic term $0.3X^3$ is represented by a higher odd power. For $Y=3+7X^7+\varepsilon$, both forms of BIC choose the true model, with coefficients $2.959$ and $7.001$, while $C_p$ and $C_p^{\mathrm F}$ add $X^2$, ISLR's BIC choosing no more predictors than its $C_p$ as Proposition 5.14(ii) requires; $R_a^2$ chooses four predictors. ■

**Exercise 5.5** (ISLR §6.6, exercise 9, parts (a), (b), (e)–(g)). Predict `Apps` from the other variables of `College`. (a) Split the data into a training set and a test set. (b) Fit least squares on the training set and report the test error. (e) Fit PCR with $M$ chosen by cross-validation and report the test error and $M$. (f) The same for PLS. (g) How accurately can the number of applications be predicted, and do the test errors differ much?

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
set.seed(1); tr9 <- sample(nrow(College), nrow(College) / 2); te9 <- College[-tr9, ]
ls9 <- lm(Apps ~ ., data = College, subset = tr9)
mse9 <- function(pred) mean((te9$Apps - pred)^2)
set.seed(1); pcr9 <- pcr(Apps ~ ., data = College, subset = tr9, scale = TRUE, validation = "CV")
set.seed(1); pls9 <- plsr(Apps ~ ., data = College, subset = tr9, scale = TRUE, validation = "CV")
cv9 <- rbind(PCR = MSEP(pcr9, estimate = "CV")$val[1, 1, -1], PLS = MSEP(pls9, estimate = "CV")$val[1, 1, -1])
round(cv9 / 1e6, 3)
M9 <- apply(cv9, 1, which.min); M9
c(onesigma_PCR = selectNcomp(pcr9, method = "onesigma"), onesigma_PLS = selectNcomp(pls9, method = "onesigma"))
test9 <- c(LS = mse9(predict(ls9, te9)), PCR = mse9(drop(predict(pcr9, te9, ncomp = M9[["PCR"]]))),
           PLS = mse9(drop(predict(pls9, te9, ncomp = M9[["PLS"]]))),
           PLS_7 = mse9(drop(predict(pls9, te9, ncomp = 7))))
round(test9)
c(test_R2_LS = 1 - test9[["LS"]] / mean((te9$Apps - mean(College$Apps[tr9]))^2))

Cross-validation selects all $17$ components for both PCR and PLS, so by Theorem 5.28(v) and Proposition 5.30(iv) both fits are the least squares fit, and the three test errors coincide at $1{,}135{,}758$. The PLS cross-validation error is nearly flat from seven directions ($1.714$ million against $1.612$ at $17$), and the one-sigma rule of `selectNcomp`, a version of the one-standard-error rule, selects $7$, with the lower test error $1{,}095{,}359$; PCR needs $16$ components to come close. With a test $R^2$ of $0.9031$ the number of applications is predicted well, and the approaches differ little once $M$ is large. ■

**Exercise 5.6** (ISLR §6.6, exercise 10). (a) Generate $p=20$ features, $n=1000$ observations and $Y=X\beta+\varepsilon$ with some $\beta_j=0$. (b) Split into $100$ training and $900$ test observations. (c), (d) Perform best subset selection on the training set and plot the training and test MSE of the best model of each size. (e) Where is the test MSE smallest? (f) How does that model compare with the true one? (g) Plot $\sqrt{\sum_j(\beta_j-\hat\beta_j^r)^2}$ against the size $r$ and compare with (d).

**Solution.** Five of the twenty coefficients are set to zero:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.3)
set.seed(1); X10 <- matrix(rnorm(1000 * 20), 1000); colnames(X10) <- paste0("x", 1:20)
beta10 <- rnorm(20); beta10[c(3, 4, 9, 10, 19)] <- 0
Y10 <- drop(X10 %*% beta10 + rnorm(1000)); tr10 <- sample(1000, 100)
f10 <- regsubsets(x = X10[tr10, ], y = Y10[tr10], nvmax = 20)
res10 <- t(sapply(1:20, function(r) {
  cf <- coef(f10, id = r); S <- names(cf)[-1]; b <- setNames(rep(0, 20), colnames(X10)); b[S] <- cf[-1]
  c(train = mean((Y10[tr10] - cf[1] - X10[tr10, S, drop = FALSE] %*% cf[-1])^2),
    test = mean((Y10[-tr10] - cf[1] - X10[-tr10, S, drop = FALSE] %*% cf[-1])^2),
    coef_dist = sqrt(sum((beta10 - b)^2))) }))
round(t(res10), 3)
c(test_min = which.min(res10[, "test"]), dist_min = which.min(res10[, "coef_dist"]))
setdiff(colnames(X10)[beta10 != 0], names(coef(f10, 15))[-1]); setdiff(names(coef(f10, 15))[-1], colnames(X10)[beta10 != 0])
round(beta10, 3); round(coef(f10, 15)["x4"], 3)
op <- par(mfrow = c(1, 3), mar = c(3.2, 3.8, 0.6, 0.4), mgp = c(2.2, 0.6, 0))
for (v in colnames(res10)) plot(res10[, v], type = "b", pch = 16, cex = 0.6, xlab = "size r", ylab = v)
par(op)

*Figure 5.13.* Training MSE, test MSE and coefficient distance of the best model of each size.

The training MSE decreases at every size (Proposition 5.5(iv)); the test MSE is smallest, $1.285$, at $15$ predictors, the number of nonzero coefficients, and the coefficient distance is smallest, $0.467$, at the same size. The $15$-predictor model contains $14$ of the $15$ true predictors and `x4`, whose coefficient is $0$ and receives $0.203$, in place of `x1`, whose coefficient $0.235$ is the smallest nonzero one in absolute value. The curves of (d) and (g) have the same shape: the estimates closest to $\beta$ predict best. ■

**Exercise 5.7** (ISLR §6.6, exercise 11, with best subset selection and PCR). Predict the per capita crime rate `crim` of `Boston`. (a) Try best subset selection and PCR and present the results. (b) Propose a model, evaluated by cross-validation rather than training error. (c) Does it involve all the features?

**Solution.** Ten-fold cross-validation, with the search inside each training set for best subset selection:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
pB <- ncol(Boston) - 1; set.seed(1); folds_B <- sample(rep(1:10, length = nrow(Boston)))
mse_B <- sapply(1:10, function(j) {
  fj <- regsubsets(crim ~ ., data = Boston[folds_B != j, ], nvmax = pB)
  mm <- model.matrix(crim ~ ., Boston[folds_B == j, ])
  sapply(1:pB, function(i) { cf <- coef(fj, id = i); mean((Boston$crim[folds_B == j] - mm[, names(cf)] %*% cf)^2) }) })
cv_B <- rowMeans(mse_B); se_B <- apply(mse_B, 1, sd) / sqrt(10)
round(rbind(CV = cv_B, SE = se_B), 2)
c(CV_min = which.min(cv_B), one_SE = min(which(cv_B <= min(cv_B) + se_B[which.min(cv_B)])))
fB <- regsubsets(crim ~ ., data = Boston, nvmax = pB)
names(coef(fB, which.min(cv_B)))[-1]; coef(fB, 1)
set.seed(1); pcr_B <- pcr(crim ~ ., data = Boston, scale = TRUE, validation = "CV")
round(MSEP(pcr_B, estimate = "CV")$val[1, 1, ], 2)
c(PCR_min = which.min(MSEP(pcr_B, estimate = "CV")$val[1, 1, ]) - 1,
  onesigma = selectNcomp(pcr_B, method = "onesigma"), var_crim = var(Boston$crim))
summary(Boston$crim)

The best subset cross-validation errors fall from $46.01$ with one predictor to a minimum of $42.68$ with eleven, but their standard errors are $12.88$ to $13.93$, as `crim` is highly skewed (median $0.25651$, mean $3.61352$, maximum $88.97620$), and the one-standard-error rule chooses the single predictor `rad`. PCR's error falls from $74.13$ with no component to $53.52$ with one and is smallest, $43.65$, with all twelve, which is least squares; the one-sigma rule chooses four components. (b) The regression on `rad` alone: no larger model lowers the estimated test error by more than one standard error. (c) It involves one of the twelve features, because the others add less to prediction than the uncertainty of the estimate. ■

**Exercise 5.8** (ISLR §12.6, exercise 6). Suppose the first $M$ principal component score vectors are known. Using the approximation problem (ii) of Theorem 5.22 (ISLR (12.6)), show that the first $M$ loading vectors can be obtained by $p$ separate least squares regressions, each with the score vectors as predictors and one column of $\mathbf{X}$ as response.

**Solution.** Let $M\leqslant r$ and fix $\mathbf{A}=\mathbf{Z}_M$. The objective separates by column: with $\boldsymbol{\beta}_j'$ row $j$ of $\mathbf{B}$, $\|\mathbf{X}-\mathbf{Z}_M\mathbf{B}'\|_F^2=\sum_j\|\mathbf{x}_j-\mathbf{Z}_M\boldsymbol{\beta}_j\|^2$, and the $j$th term is the residual sum of squares of the regression of $\mathbf{x}_j$ on the columns of $\mathbf{Z}_M$ without intercept, a full-rank design since $\mathbf{Z}_M'\mathbf{Z}_M=n\operatorname{diag}(\lambda_1,\ldots,\lambda_M)$ with all $\lambda_m>0$ (Theorem 5.20). Its minimiser is $\hat{\boldsymbol{\beta}}_j=(\mathbf{Z}_M'\mathbf{Z}_M)^{-1}\mathbf{Z}_M'\mathbf{x}_j$. Now $\mathbf{Z}_M'\mathbf{X}=\boldsymbol{\Phi}_M'\mathbf{X}'\mathbf{X}=n\boldsymbol{\Phi}_M'\mathbf{S} =n\operatorname{diag}(\lambda_1,\ldots,\lambda_M)\boldsymbol{\Phi}_M'$, because $\mathbf{S}\mathbf{v}_m=\lambda_m\mathbf{v}_m$; so $\mathbf{Z}_M'\mathbf{x}_j=n\operatorname{diag}(\lambda_1,\ldots,\lambda_M)(\phi_{j1},\ldots,\phi_{jM})'$ and $\hat{\boldsymbol{\beta}}_j=(\phi_{j1},\ldots,\phi_{jM})'$: the regression of the $j$th feature on the scores returns the $j$th loadings. On `USArrests` with $M=2$: ■

In [ ]:
Z2 <- pr$x[, 1:2]; Xs_us <- scale(USArrests)
B_reg <- t(sapply(colnames(Xs_us), function(v) coef(lm(Xs_us[, v] ~ Z2 - 1))))
signif(max(abs(B_reg - pr$rotation[, 1:2])), 3)

**Exercise 5.9** (ISLR §12.6, exercise 8). On `USArrests`, calculate the proportion of variance explained (a) from the `sdev` output of `prcomp`, and (b) from ISLR (12.10), using the loadings of `prcomp` and the same centred and scaled data.

**Solution.** By Proposition 5.21 both compute $\lambda_m/\sum_l\lambda_l$, the divisor of the variances cancelling:

In [ ]:
pve_a <- pr$sdev^2 / sum(pr$sdev^2)
Xs_us <- scale(USArrests)
pve_b <- colSums((Xs_us %*% pr$rotation)^2) / sum(Xs_us^2)
round(rbind(pve_a, pve_b), 4); signif(max(abs(pve_a - pve_b)), 3)

The two agree to rounding error: $0.6201$, $0.2474$, $0.0891$ and $0.0434$. ■

**Exercise 5.10** (SOA sample question 5). Which statements are correct? I. Principal components analysis provides low-dimensional linear surfaces that are closest to the observations. II. The first principal component is the line in $p$-dimensional space that is closest to the observations. III. PCA finds a low-dimensional representation of a data set that contains as much variation as possible. IV. PCA serves as a tool for data visualisation. (A) I, II and III only; (B) I, II and IV only; (C) I, III and IV only; (D) II, III and IV only; (E) I, II, III and IV.

**Solution.** I: by Theorem 5.22(iv), $\bar{\mathbf{x}}+\mathcal{C}(\boldsymbol{\Phi}_M)$ is the affine $M$-dimensional subspace with the smallest sum of squared distances to the observations. II: the case $M=1$, the line being the only closest one when $\lambda_1>\lambda_2$. III: by Theorem 5.22(i), the total variance $\operatorname{tr}(\mathbf{Q}'\mathbf{S}\mathbf{Q})$ of the scores on any $M$ orthonormal directions is at most $\lambda_1+\cdots+\lambda_M$, attained by the first $M$ loading vectors. IV: a plot of the first two score vectors, with the loadings in a biplot, is a two-dimensional view of the data (Example 5.5). All four are correct: (E). ■

**Exercise 5.11** (SOA sample question 6). Which statements are correct? I. The proportion of variance explained by an additional principal component never decreases as more principal components are added. II. The cumulative proportion of variance explained never decreases as more principal components are added. III. Using all possible principal components provides the best understanding of the data. IV. A scree plot provides a method for determining the number of principal components to use. (A) I and II only; (B) I and III only; (C) I and IV only; (D) II and III only; (E) II and IV only.

**Solution.** I is false and II true, by Proposition 5.21: $\mathrm{PVE}_m$ is nonincreasing in $m$, strictly decreasing whenever $\lambda_m>\lambda_{m+1}$ (on `USArrests`, $0.620$, $0.247$, $0.089$, $0.043$), and the cumulative sum adds nonnegative terms. III is false: with all components the approximation of Theorem 5.22(ii) is exact, so nothing is summarised, whereas the aim is the smallest number of components that gives a good description. IV is true: the scree plot, read at its elbow, is the usual method for visualisation (ISLR §12.2.4). The answer is (E). ■

**Exercise 5.12** (SOA sample question 8). Which statements describe advantages of a subset selection process, such as backward selection? I. It will likely result in a simpler model. II. It will likely improve prediction accuracy. III. The results are likely to be easier to interpret. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) the correct answer is not given by (A), (B), (C) or (D).

**Solution.** I and III: the selected model has at most as many predictors as the full one, and fewer coefficients to interpret. II: by Corollary 5.9, deleting a predictor whose coefficient is zero lowers the in-sample prediction error by $\sigma^2/n$, and deleting one whose contribution to the squared bias is below $\sigma^2$ still lowers it; the selection criteria are built to delete such predictors (Theorem 5.12), so accuracy is likely, though not certain, to improve. All three hold: (D). ■

**Exercise 5.13** (SOA sample question 30). Principal components analysis is applied to a large data set with four variables, and the loadings of the four principal components are estimated. Which are true? I. The loadings are unique. II. For a given principal component, the sum of the squares of the loadings across the four variables is one. III. Together, the four principal components explain $100\%$ of the variance. (A) None; (B) I and II only; (C) I and III only; (D) II and III only; (E) the correct answer is not given by (A), (B), (C) or (D).

**Solution.** I is false: $\boldsymbol{\phi}_m$ and $-\boldsymbol{\phi}_m$ solve the same problem, so the loadings are at best unique up to sign (Theorem 5.20(v)). II is true: the loading vectors are unit vectors by Definition 5.19. III is true: with $M=p=4$, $\mathrm{PVE}_1+\cdots+\mathrm{PVE}_4=\sum_{m\leqslant4}\lambda_m/\operatorname{tr}(\mathbf{S})=1$ by Proposition 5.21, for any data with positive total variance. The answer is (D). ■

**Exercise 5.14** (SOA sample question 35). A scree plot of five principal components shows proportions of variance explained of about $0.61$, $0.23$, $0.09$, $0.05$ and $0.02$. What is the minimum number of principal components needed to explain at least $80\%$ of the variance? (A) one; (B) two; (C) three; (D) four; (E) it cannot be determined from the information given.

**Solution.** By Proposition 5.21 the cumulative proportion after $M$ components is the sum of the first $M$ proportions, so it can be determined, and (E) is excluded:

In [ ]:
cumsum(c(0.61, 0.23, 0.09, 0.05, 0.02))

One component explains $0.61<0.80$ and two explain $0.84\geqslant0.80$; the reading of the plot is not critical, since the first proportion is visibly below $0.8$ and the first two sum to well above it. The answer is (B). ■

**Exercise 5.15** (SOA sample question 37). Analysts W, X, Y and Z perform principal components analysis on the same data set with three variables, using different programs with their default settings, and obtain the first loading vectors W $(-0.549,-0.594,0.587)$, X $(-0.549,0.594,0.587)$, Y $(0.549,-0.594,-0.587)$, Z $(0.140,-0.570,-0.809)$. Which are plausible explanations? I. Loadings are unique up to a sign flip, so the programs of X and Y could make different arbitrary sign choices. II. Z's program does not scale the variables by default while Y's does. III. Loadings are unique up to a sign flip, so the programs of W and X could make different arbitrary sign choices. (A) None; (B) I and II only; (C) I and III only; (D) II and III only; (E) the correct answer is not given by (A), (B), (C) or (D).

**Solution.** I is plausible: Y's vector is $-1$ times X's, and by Theorem 5.20(v) a first loading vector with a simple eigenvalue is determined up to that sign. III is not: W and X differ in one coordinate only, and neither is $-1$ times the other, so they cannot both be the first loading vector of a matrix with a simple first eigenvalue (Theorem 5.20(v)); a sign choice flips the whole vector. II is plausible: by Proposition 5.24 the components of the unscaled data are those of $\mathbf{S}$ and of the scaled data those of the correlation matrix, which have different eigenvectors in general, and Z's vector is not $\pm$ Y's. The answer is (B). ■

**Exercise 5.16** (SOA sample question 54). A regression of executive compensation gives the estimates, standard errors, $t$ statistics and $p$-values: intercept $-28{,}595.5$, $220.5$, $-129.7$, ${<}0.001$; `AGEMINUS35` $7{,}366.3$, $12.5$, $588.1$, ${<}0.001$; `TOPSCHOOL` $50.0$, $119.7$, $0.4$, $0.676$; `LARGECITY` $147.9$, $119.7$, $1.2$, $0.217$; `MBA` $2{,}490.9$, $119.7$, $20.8$, ${<}0.001$; `YEARSEXP` $15{,}286.6$, $7.2$, $2{,}132.8$, ${<}0.001$. The significance level is $\alpha=0.10$. Which predictor or predictors should be removed first before rerunning the model? (A) `AGEMINUS35`; (B) `AGEMINUS35`, `MBA` and `YEARSEXP`; (C) `TOPSCHOOL`; (D) `TOPSCHOOL` and `LARGECITY`; (E) `YEARSEXP`.

**Solution.** Backward selection by $t$ ratios (Definition 5.3) deletes one predictor at a time, the one with the smallest $|t|$ and largest $p$-value, provided its $p$-value exceeds $\alpha$; by Proposition 5.4(ii) it is also the deletion that increases $\mathrm{RSS}$ least. `TOPSCHOOL` has $p=0.676>0.10$, the largest. (A), (B) and (E) delete predictors with $p<0.001$. (D) deletes two at once, but after `TOPSCHOOL` is removed the $t$ ratio of `LARGECITY` changes, since each $t$ ratio depends on the other predictors in the model (Unit 3's theorem **Frisch–Waugh–Lovell**), and it may become significant. The answer is (C). ■

**Exercise 5.17** (SOA sample question 61). A linear model is fitted with six predictors F, G, H, I, J and K. Which statements about using AIC or BIC to select the predictors are true? I. AIC and BIC provide a direct estimate of the test error. II. When choosing between the subsets $\{F,G,H\}$ and $\{I,J,K\}$, AIC will always select the same subset as BIC. III. For $n>7$, the number of variables selected by BIC is at most the number selected by AIC. (A) None; (B) I and II only; (C) I and III only; (D) II and III only; (E) the correct answer is not given by (A), (B), (C) or (D).

**Solution.** I is false: AIC and BIC adjust the training error by a penalty (Definition 5.10); by Theorem 5.12 such a criterion is an unbiased estimate of the in-sample prediction error, up to a constant, when $\hat\sigma^2$ is unbiased, which is an indirect estimate, whereas a validation set or cross-validation estimates the test error directly. II is true: both subsets have three predictors, and by Proposition 5.14(i) both criteria prefer the one with the smaller $\mathrm{RSS}$. III is true by Proposition 5.14(ii), $\ln n>2$ for $n\geqslant8$. The answer is (D). ■

**Exercise 5.18** (SOA sample question 70). A data set has $100$ observations, six predictors $X_1,\ldots,X_6$ and a response. Three least squares models give: I, $X_1$ only, $\mathrm{RSS}=183{,}663.30$, residual standard error $43.2911$; II, $X_1$ and $X_2$, $\mathrm{RSS}=8{,}826.47$, $9.5391$; III, all six, $\mathrm{RSS}=8{,}319.59$, $9.4582$. AIC is used to select among them. (A) Model I, AIC $1{,}838.42$; (B) Model I, AIC $1{,}874.12$; (C) Model II, AIC $91.84$; (D) Model II, AIC $91.90$; (E) Model III, AIC $93.93$.

**Solution.** The syllabus form of AIC (Definition 5.10) takes $\hat\sigma^2$ from the model with all six predictors, $9.4582^2$:

In [ ]:
rss70 <- c(183663.30, 8826.47, 8319.59); d70 <- c(1, 2, 6); rse70 <- c(43.2911, 9.5391, 9.4582)
round(rbind(AIC_full_sigma = (rss70 + 2 * d70 * rse70[3]^2) / 100,
            AIC_own_sigma = (rss70 + 2 * d70 * rse70^2) / 100,
            rse_from_rss = sqrt(rss70 / (100 - d70 - 1))), 4)

Model II has the smallest AIC, $91.84$: (C). The values $1{,}838.42$ and $93.93$ of (A) and (E) are the AIC of Models I and III, which are larger. The values of (B) and (D), $1{,}874.12$ and $91.90$, result from using each model's own residual variance in place of $\hat\sigma^2$; then the penalty no longer uses a common estimate of $\sigma^2$, and Proposition 5.13(ii) no longer applies. The residual standard errors of the table follow from $\mathrm{RSS}$ and $n-d-1$, as the last row shows. ■